# 02 — Model Development and Evaluation

This notebook develops and evaluates five machine-learning classifiers for predicting telecommunications business-customer churn: Logistic Regression, Decision Tree, Random Forest, XGBoost and LightGBM. The models use the prepared training and test datasets generated in Notebook 1.

Baseline and imbalance-aware configurations are examined to assess how different modelling approaches respond to the highly imbalanced churn target. Class weighting is applied to the Spark-based models, while XGBoost and LightGBM use scale_pos_weight.

Model development includes five-fold cross-validation and hyperparameter tuning, with PR-AUC used as the primary selection metric. Final configurations are evaluated on the held-out test set using accuracy, churn precision, recall, F1-score, ROC-AUC, PR-AUC and confusion-matrix counts.

The resulting model configurations and evaluation outputs are retained for the subsequent explainability and final model assessment in Notebook 3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. Environment Setup and Data Loading

The modelling environment is initialised using PySpark, and the prepared datasets generated in the preceding data-preparation notebook are loaded for model development. Separate feature representations are used for Logistic Regression and tree-based algorithms, while class-weighted training datasets are retained for imbalance-aware modelling. The held-out test data remain separate from model training and hyperparameter optimisation.

In [ ]:
# ============================================================
# Cell 1 — Environment Setup
# ============================================================

import os
import json
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Project paths
PROJECT = "/content/drive/MyDrive/7005SCN_Dissertation"
OUTPUTS = os.path.join(PROJECT, "outputs")
MODELS = os.path.join(PROJECT, "models")

# Start Spark
spark = (
    SparkSession.builder
    .appName("7005SCN_Telecom_Churn_Model_Development")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("=== Environment ready ===")
print("Spark version :", spark.version)
print("Spark UI      :", spark.sparkContext.uiWebUrl)
print("Outputs found :", os.path.exists(OUTPUTS))
print("Models found  :", os.path.exists(MODELS))

=== Environment ready ===
Spark version : 4.0.4
Spark UI      : http://433699def06f:4040
Outputs found : True
Models found  : True


In [ ]:
# ============================================================
# Cell 2 — Load and Verify Prepared Modelling Data
# ============================================================

# Load prepared datasets
train_tree = (
    spark.read
    .parquet(os.path.join(OUTPUTS, "train_tree_features"))
    .cache()
)

test_tree = (
    spark.read
    .parquet(os.path.join(OUTPUTS, "test_tree_features"))
    .cache()
)

train_tree_weighted = (
    spark.read
    .parquet(os.path.join(OUTPUTS, "train_tree_features_weighted"))
    .cache()
)

train_lr = (
    spark.read
    .parquet(os.path.join(OUTPUTS, "train_lr_features"))
    .cache()
)

test_lr = (
    spark.read
    .parquet(os.path.join(OUTPUTS, "test_lr_features"))
    .cache()
)

train_lr_weighted = (
    spark.read
    .parquet(os.path.join(OUTPUTS, "train_lr_features_weighted"))
    .cache()
)


# Load readable feature-name mapping
with open(
    os.path.join(OUTPUTS, "feature_name_mapping.json"),
    "r"
) as file:
    feature_mapping = json.load(file)

feature_names = [
    feature_mapping[str(i)]
    for i in range(len(feature_mapping))
]


# Verify row counts
dataset_counts = {
    "Tree training": train_tree.count(),
    "Tree test": test_tree.count(),
    "Weighted tree training": train_tree_weighted.count(),
    "LR training": train_lr.count(),
    "LR test": test_lr.count(),
    "Weighted LR training": train_lr_weighted.count(),
}

print("=== Prepared modelling data loaded ===\n")

for name, count in dataset_counts.items():
    print(f"{name:<27}: {count}")

print(f"\nReadable feature names     : {len(feature_names)}")


# Verify expected structure
print("\n=== Tree dataset schema ===")
train_tree.printSchema()

print("\n=== Logistic Regression dataset schema ===")
train_lr.printSchema()

print("\n=== First 5 readable feature names ===")
for i, name in enumerate(feature_names[:5]):
    print(f"{i}: {name}")

=== Prepared modelling data loaded ===

Tree training              : 6762
Tree test                  : 1691
Weighted tree training     : 6762
LR training                : 6762
LR test                    : 1691
Weighted LR training       : 6762

Readable feature names     : 24

=== Tree dataset schema ===
root
 |-- features_unscaled: vector (nullable = true)
 |-- label: double (nullable = true)


=== Logistic Regression dataset schema ===
root
 |-- features_scaled: vector (nullable = true)
 |-- label: double (nullable = true)


=== First 5 readable feature names ===
0: CRM_PID_Value_Segment_Bronze
1: CRM_PID_Value_Segment_Silver
2: CRM_PID_Value_Segment_Gold
3: CRM_PID_Value_Segment_Platinum
4: CRM_PID_Value_Segment_SME


# 2. Modelling and Evaluation Framework

Model performance is evaluated using a consistent binary-classification framework. Because the target variable is substantially imbalanced, with churn representing only 6.49% of observations, accuracy alone may provide an incomplete assessment of predictive performance.

Accordingly, evaluation includes accuracy, ROC-AUC and PR-AUC alongside precision, recall and F1-score calculated specifically for the churn class (label = 1). Confusion-matrix values are also retained to quantify true positives, false positives, true negatives and false negatives.

Hyperparameter optimisation will be conducted using cross-validation on the training data only. The held-out test set will not be used during model fitting or hyperparameter selection and will be reserved for final evaluation.

In [ ]:
# ============================================================
# Cell 3 — Modelling and Evaluation Framework
# ============================================================

from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F


# ------------------------------------------------------------
# Binary classification evaluators
# ------------------------------------------------------------

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)


# ------------------------------------------------------------
# Reusable model evaluation function
# ------------------------------------------------------------

def evaluate_binary_model(predictions, model_name):
    """
    Evaluate binary churn predictions.

    Positive class:
        label = 1.0 -> Churn

    Returns:
        Dictionary containing accuracy, churn-class precision,
        recall, F1, ROC-AUC, PR-AUC and confusion-matrix counts.
    """

    # Confusion-matrix counts
    counts = (
        predictions
        .select("label", "prediction")
        .groupBy("label", "prediction")
        .count()
        .collect()
    )

    confusion = {
        (float(row["label"]), float(row["prediction"])): row["count"]
        for row in counts
    }

    tn = confusion.get((0.0, 0.0), 0)
    fp = confusion.get((0.0, 1.0), 0)
    fn = confusion.get((1.0, 0.0), 0)
    tp = confusion.get((1.0, 1.0), 0)

    total = tn + fp + fn + tp

    # Accuracy
    accuracy = (
        (tp + tn) / total
        if total > 0 else 0.0
    )

    # Churn-class precision
    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0 else 0.0
    )

    # Churn-class recall
    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0 else 0.0
    )

    # Churn-class F1
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    # Ranking metrics
    roc_auc = roc_evaluator.evaluate(predictions)
    pr_auc = pr_evaluator.evaluate(predictions)

    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Churn Precision": precision,
        "Churn Recall": recall,
        "Churn F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    return results


print("=== Evaluation framework ready ===")
print("Positive class: Churn (label = 1.0)")
print(
    "Metrics: Accuracy, Churn Precision, Churn Recall, "
    "Churn F1, ROC-AUC, PR-AUC"
)

=== Evaluation framework ready ===
Positive class: Churn (label = 1.0)
Metrics: Accuracy, Churn Precision, Churn Recall, Churn F1, ROC-AUC, PR-AUC


# 3. Logistic Regression

## 3.1 Baseline Logistic Regression

An untreated Logistic Regression model is first trained to establish a baseline for comparison with subsequent imbalance-aware modelling. The model uses the scaled feature representation prepared using training data only and is fitted without class weighting or hyperparameter optimisation.

This baseline provides a reference for assessing the effect of class weighting on the identification of churn customers. Model performance is evaluated on the held-out test set using the common evaluation framework defined previously.

In [ ]:
# ============================================================
# Cell 4 — Baseline Logistic Regression
# ============================================================

from pyspark.ml.classification import LogisticRegression

# Initialise baseline Logistic Regression
lr_baseline = LogisticRegression(
    featuresCol="features_scaled",
    labelCol="label",
    maxIter=100
)

# Train model
start_time = time.time()

lr_baseline_model = lr_baseline.fit(train_lr)

lr_baseline_training_time = time.time() - start_time


# Generate predictions on held-out test data
lr_baseline_predictions = lr_baseline_model.transform(test_lr)


# Evaluate using the common evaluation framework
lr_baseline_results = evaluate_binary_model(
    lr_baseline_predictions,
    "Logistic Regression — Baseline"
)


# Add training time to results
lr_baseline_results["Training Time (s)"] = (
    lr_baseline_training_time
)


# Display results
print("=== Baseline Logistic Regression ===\n")

for metric in [
    "Accuracy",
    "Churn Precision",
    "Churn Recall",
    "Churn F1",
    "ROC-AUC",
    "PR-AUC"
]:
    print(
        f"{metric:<18}: "
        f"{lr_baseline_results[metric]:.4f}"
    )

print("\nConfusion Matrix Counts")
print(
    f"TN: {lr_baseline_results['TN']}   "
    f"FP: {lr_baseline_results['FP']}"
)
print(
    f"FN: {lr_baseline_results['FN']}   "
    f"TP: {lr_baseline_results['TP']}"
)

print(
    f"\nTraining time: "
    f"{lr_baseline_training_time:.2f} seconds"
)

=== Baseline Logistic Regression ===

Accuracy          : 0.9349
Churn Precision   : 0.0000
Churn Recall      : 0.0000
Churn F1          : 0.0000
ROC-AUC           : 0.6003
PR-AUC            : 0.0918

Confusion Matrix Counts
TN: 1581   FP: 0
FN: 110   TP: 0

Training time: 6.88 seconds


### Baseline Logistic Regression Interpretation

The untreated Logistic Regression model achieved 93.49% accuracy but classified every test observation as non-churn, failing to identify any of the 110 churn customers. Although ROC-AUC was 0.6003 and PR-AUC was 0.0918, churn precision, recall and F1-score were all zero. This demonstrates why accuracy alone is insufficient for evaluating performance on this imbalanced dataset.

## 3.2 Class-Weighted Logistic Regression

To address the substantial imbalance between churn and non-churn observations, a class-weighted Logistic Regression model is trained using the balanced class weights calculated from the training data during preprocessing. Greater weight is assigned to the minority churn class so that classification errors involving churn observations exert greater influence during model fitting.

The model uses the same scaled predictors and modelling configuration as the untreated baseline, allowing the effect of class weighting to be examined while holding the remaining modelling conditions constant. Evaluation is performed using the same naturally distributed held-out test set; class weights are applied only during training.

In [ ]:
# ============================================================
# Cell 5 — Class-Weighted Logistic Regression
# ============================================================

from pyspark.ml.classification import LogisticRegression

# Initialise class-weighted Logistic Regression
lr_weighted = LogisticRegression(
    featuresCol="features_scaled",
    labelCol="label",
    weightCol="classWeight",
    maxIter=100
)

# Train model
start_time = time.time()

lr_weighted_model = lr_weighted.fit(train_lr_weighted)

lr_weighted_training_time = time.time() - start_time


# Generate predictions on the natural, unweighted held-out test set
lr_weighted_predictions = lr_weighted_model.transform(test_lr)


# Evaluate using the common evaluation framework
lr_weighted_results = evaluate_binary_model(
    lr_weighted_predictions,
    "Logistic Regression — Class Weighted"
)


# Add training time
lr_weighted_results["Training Time (s)"] = (
    lr_weighted_training_time
)


# Display results
print("=== Class-Weighted Logistic Regression ===\n")

for metric in [
    "Accuracy",
    "Churn Precision",
    "Churn Recall",
    "Churn F1",
    "ROC-AUC",
    "PR-AUC"
]:
    print(
        f"{metric:<18}: "
        f"{lr_weighted_results[metric]:.4f}"
    )

print("\nConfusion Matrix Counts")
print(
    f"TN: {lr_weighted_results['TN']}   "
    f"FP: {lr_weighted_results['FP']}"
)
print(
    f"FN: {lr_weighted_results['FN']}   "
    f"TP: {lr_weighted_results['TP']}"
)

print(
    f"\nTraining time: "
    f"{lr_weighted_training_time:.2f} seconds"
)

=== Class-Weighted Logistic Regression ===

Accuracy          : 0.5967
Churn Precision   : 0.0831
Churn Recall      : 0.5182
Churn F1          : 0.1432
ROC-AUC           : 0.5982
PR-AUC            : 0.0918

Confusion Matrix Counts
TN: 952   FP: 629
FN: 53   TP: 57

Training time: 2.58 seconds


### Class-Weighted Logistic Regression Interpretation

Applying class weighting substantially changed the classification behaviour of Logistic Regression. Churn recall increased from 0 to 51.82%, with the weighted model correctly identifying 57 of the 110 churn customers in the held-out test set. Churn F1-score consequently increased from 0 to 0.1432.

This improvement in minority-class detection was accompanied by a substantial increase in false-positive predictions. The model incorrectly classified 629 non-churn customers as churn, resulting in churn precision of 8.31% and reducing overall accuracy from 93.49% to 59.67%.

ROC-AUC and PR-AUC remained broadly unchanged at approximately 0.60 and 0.09 respectively. This indicates that class weighting primarily altered the model's classification behaviour rather than materially improving its underlying ranking ability. The results demonstrate the trade-off between identifying a greater proportion of churn customers and generating additional false-positive churn predictions.

## 3.3 Hyperparameter Tuning and Cross-Validation

Hyperparameter optimisation is performed using cross-validation on the training data only. A compact parameter grid is used to evaluate alternative regularisation strengths and Elastic Net configurations while limiting unnecessary computational complexity.

Five-fold cross-validation is applied, consistent with the taught modelling approach, with PR-AUC used as the optimisation metric. PR-AUC is selected because churn represents a small minority of observations and the metric focuses more directly on performance for the positive class. The held-out test set is not used during hyperparameter selection.

In [ ]:
# ============================================================
# Cell 6 — Baseline Logistic Regression Cross-Validation
# ============================================================

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# ------------------------------------------------------------
# Logistic Regression estimator
# ------------------------------------------------------------

lr_cv_baseline = LogisticRegression(
    featuresCol="features_scaled",
    labelCol="label",
    maxIter=100
)

# ------------------------------------------------------------
# Compact hyperparameter grid
# ------------------------------------------------------------

lr_param_grid = (
    ParamGridBuilder()
    .addGrid(
        lr_cv_baseline.regParam,
        [0.01, 0.1]
    )
    .addGrid(
        lr_cv_baseline.elasticNetParam,
        [0.0, 0.5]
    )
    .build()
)

# ------------------------------------------------------------
# Cross-validation evaluator
# Optimisation metric: PR-AUC
# ------------------------------------------------------------

lr_cv_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

# ------------------------------------------------------------
# Five-fold cross-validation
# ------------------------------------------------------------

lr_baseline_cv = CrossValidator(
    estimator=lr_cv_baseline,
    estimatorParamMaps=lr_param_grid,
    evaluator=lr_cv_evaluator,
    numFolds=5,
    seed=42,
    parallelism=2
)

print("=== Baseline Logistic Regression CV ===")
print(f"Parameter combinations : {len(lr_param_grid)}")
print("Cross-validation folds : 5")
print(
    f"Total model fits       : "
    f"{len(lr_param_grid) * 5}"
)
print("Optimisation metric    : PR-AUC")
print("\nStarting cross-validation...\n")

# ------------------------------------------------------------
# Fit using TRAINING DATA ONLY
# ------------------------------------------------------------

start_time = time.time()

lr_baseline_cv_model = lr_baseline_cv.fit(train_lr)

lr_baseline_cv_time = time.time() - start_time

# ------------------------------------------------------------
# Extract best model and parameters
# ------------------------------------------------------------

lr_baseline_best_model = lr_baseline_cv_model.bestModel

best_reg_param = (
    lr_baseline_best_model.getRegParam()
)

best_elastic_net = (
    lr_baseline_best_model.getElasticNetParam()
)

best_cv_pr_auc = max(
    lr_baseline_cv_model.avgMetrics
)

# ------------------------------------------------------------
# Display cross-validation results
# ------------------------------------------------------------

print("=== Baseline LR Cross-Validation Complete ===\n")

print(
    f"Best regParam         : "
    f"{best_reg_param}"
)

print(
    f"Best elasticNetParam  : "
    f"{best_elastic_net}"
)

print(
    f"Best mean CV PR-AUC   : "
    f"{best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time : "
    f"{lr_baseline_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for params, metric in zip(
    lr_baseline_cv_model.getEstimatorParamMaps(),
    lr_baseline_cv_model.avgMetrics
):
    print(
        f"regParam={params[lr_cv_baseline.regParam]:<5} | "
        f"elasticNet={params[lr_cv_baseline.elasticNetParam]:<3} | "
        f"Mean CV PR-AUC={metric:.4f}"
    )

=== Baseline Logistic Regression CV ===
Parameter combinations : 4
Cross-validation folds : 5
Total model fits       : 20
Optimisation metric    : PR-AUC

Starting cross-validation...

=== Baseline LR Cross-Validation Complete ===

Best regParam         : 0.01
Best elasticNetParam  : 0.5
Best mean CV PR-AUC   : 0.0848
Cross-validation time : 19.13 seconds

All parameter combinations:
regParam=0.01  | elasticNet=0.0 | Mean CV PR-AUC=0.0807
regParam=0.01  | elasticNet=0.5 | Mean CV PR-AUC=0.0848
regParam=0.1   | elasticNet=0.0 | Mean CV PR-AUC=0.0826
regParam=0.1   | elasticNet=0.5 | Mean CV PR-AUC=0.0648


## 3.3.2 Class-Weighted Logistic Regression Cross-Validation

The class-weighted Logistic Regression model is optimised using the same hyperparameter grid, five-fold cross-validation procedure and PR-AUC evaluation criterion as the untreated model. Maintaining identical optimisation conditions allows the effect of class weighting to be compared without introducing differences in the tuning procedure.

Class weights are applied only to the training observations through the previously calculated classWeight variable. The held-out test set remains excluded from cross-validation and hyperparameter selection.

In [ ]:
# ============================================================
# Cell 7 — Class-Weighted Logistic Regression Cross-Validation
# ============================================================

# ------------------------------------------------------------
# Weighted Logistic Regression estimator
# ------------------------------------------------------------

lr_cv_weighted = LogisticRegression(
    featuresCol="features_scaled",
    labelCol="label",
    weightCol="classWeight",
    maxIter=100
)

# ------------------------------------------------------------
# Same compact hyperparameter grid
# ------------------------------------------------------------

lr_weighted_param_grid = (
    ParamGridBuilder()
    .addGrid(
        lr_cv_weighted.regParam,
        [0.01, 0.1]
    )
    .addGrid(
        lr_cv_weighted.elasticNetParam,
        [0.0, 0.5]
    )
    .build()
)

# ------------------------------------------------------------
# Five-fold cross-validation
# Same PR-AUC optimisation criterion
# ------------------------------------------------------------

lr_weighted_cv = CrossValidator(
    estimator=lr_cv_weighted,
    estimatorParamMaps=lr_weighted_param_grid,
    evaluator=lr_cv_evaluator,
    numFolds=5,
    seed=42,
    parallelism=2
)

print("=== Class-Weighted Logistic Regression CV ===")
print(f"Parameter combinations : {len(lr_weighted_param_grid)}")
print("Cross-validation folds : 5")
print(
    f"Total model fits       : "
    f"{len(lr_weighted_param_grid) * 5}"
)
print("Optimisation metric    : PR-AUC")
print("\nStarting cross-validation...\n")

# ------------------------------------------------------------
# Fit using WEIGHTED TRAINING DATA ONLY
# ------------------------------------------------------------

start_time = time.time()

lr_weighted_cv_model = lr_weighted_cv.fit(
    train_lr_weighted
)

lr_weighted_cv_time = time.time() - start_time

# ------------------------------------------------------------
# Extract best model and parameters
# ------------------------------------------------------------

lr_weighted_best_model = (
    lr_weighted_cv_model.bestModel
)

weighted_best_reg_param = (
    lr_weighted_best_model.getRegParam()
)

weighted_best_elastic_net = (
    lr_weighted_best_model.getElasticNetParam()
)

weighted_best_cv_pr_auc = max(
    lr_weighted_cv_model.avgMetrics
)

# ------------------------------------------------------------
# Display cross-validation results
# ------------------------------------------------------------

print(
    "=== Class-Weighted LR "
    "Cross-Validation Complete ===\n"
)

print(
    f"Best regParam         : "
    f"{weighted_best_reg_param}"
)

print(
    f"Best elasticNetParam  : "
    f"{weighted_best_elastic_net}"
)

print(
    f"Best mean CV PR-AUC   : "
    f"{weighted_best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time : "
    f"{lr_weighted_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for params, metric in zip(
    lr_weighted_cv_model.getEstimatorParamMaps(),
    lr_weighted_cv_model.avgMetrics
):
    print(
        f"regParam="
        f"{params[lr_cv_weighted.regParam]:<5} | "
        f"elasticNet="
        f"{params[lr_cv_weighted.elasticNetParam]:<3} | "
        f"Mean CV PR-AUC={metric:.4f}"
    )

=== Class-Weighted Logistic Regression CV ===
Parameter combinations : 4
Cross-validation folds : 5
Total model fits       : 20
Optimisation metric    : PR-AUC

Starting cross-validation...

=== Class-Weighted LR Cross-Validation Complete ===

Best regParam         : 0.1
Best elasticNetParam  : 0.5
Best mean CV PR-AUC   : 0.0857
Cross-validation time : 23.51 seconds

All parameter combinations:
regParam=0.01  | elasticNet=0.0 | Mean CV PR-AUC=0.0806
regParam=0.01  | elasticNet=0.5 | Mean CV PR-AUC=0.0845
regParam=0.1   | elasticNet=0.0 | Mean CV PR-AUC=0.0814
regParam=0.1   | elasticNet=0.5 | Mean CV PR-AUC=0.0857


### Logistic Regression Cross-Validation Interpretation

Cross-validation selected different optimal regularisation settings for the untreated and class-weighted Logistic Regression models. The baseline model achieved its highest mean cross-validation PR-AUC of 0.0848 using regParam = 0.01 and elasticNetParam = 0.5, whereas the class-weighted model achieved its highest mean PR-AUC of 0.0857 using stronger regularisation (regParam = 0.1) with the same Elastic Net setting of 0.5.

The difference in the best mean cross-validation PR-AUC values was small, indicating that class weighting did not substantially improve ranking performance during cross-validation. However, the different selected regularisation strengths demonstrate that weighting altered the model optimisation process. Both sets of hyperparameters were selected exclusively through cross-validation on the training data, without reference to the held-out test set.

## 3.4 Tuned Logistic Regression Evaluation

Following hyperparameter selection through five-fold cross-validation, the best untreated and class-weighted Logistic Regression models are evaluated on the held-out test set. This represents the first use of the test data following hyperparameter optimisation.

Both models are evaluated against the same naturally imbalanced test set using accuracy, churn-class precision, recall and F1-score, ROC-AUC, PR-AUC and confusion-matrix counts. This enables the effect of class weighting to be assessed after independent hyperparameter optimisation under identical cross-validation conditions.

In [ ]:
# ============================================================
# Cell 8 — Tuned Logistic Regression Test Evaluation
# ============================================================

# ------------------------------------------------------------
# Generate held-out test predictions
# ------------------------------------------------------------

lr_tuned_baseline_predictions = (
    lr_baseline_best_model.transform(test_lr)
)

lr_tuned_weighted_predictions = (
    lr_weighted_best_model.transform(test_lr)
)


# ------------------------------------------------------------
# Evaluate both tuned models
# ------------------------------------------------------------

lr_tuned_baseline_results = evaluate_binary_model(
    lr_tuned_baseline_predictions,
    "Logistic Regression — Tuned Baseline"
)

lr_tuned_weighted_results = evaluate_binary_model(
    lr_tuned_weighted_predictions,
    "Logistic Regression — Tuned Class Weighted"
)


# ------------------------------------------------------------
# Retain CV computational information
# ------------------------------------------------------------

lr_tuned_baseline_results["CV Time (s)"] = (
    lr_baseline_cv_time
)

lr_tuned_weighted_results["CV Time (s)"] = (
    lr_weighted_cv_time
)

lr_tuned_baseline_results["Best CV PR-AUC"] = (
    best_cv_pr_auc
)

lr_tuned_weighted_results["Best CV PR-AUC"] = (
    weighted_best_cv_pr_auc
)


# ------------------------------------------------------------
# Display tuned model comparison
# ------------------------------------------------------------

print("=== Tuned Logistic Regression Test Evaluation ===\n")

print(
    f"{'Metric':<20}"
    f"{'Baseline':>14}"
    f"{'Weighted':>14}"
)

print("-" * 48)

for metric in [
    "Accuracy",
    "Churn Precision",
    "Churn Recall",
    "Churn F1",
    "ROC-AUC",
    "PR-AUC"
]:
    print(
        f"{metric:<20}"
        f"{lr_tuned_baseline_results[metric]:>14.4f}"
        f"{lr_tuned_weighted_results[metric]:>14.4f}"
    )


# ------------------------------------------------------------
# Confusion matrices
# ------------------------------------------------------------

print("\n=== Confusion Matrix Counts ===\n")

print("Tuned Baseline")
print(
    f"TN: {lr_tuned_baseline_results['TN']}   "
    f"FP: {lr_tuned_baseline_results['FP']}"
)
print(
    f"FN: {lr_tuned_baseline_results['FN']}   "
    f"TP: {lr_tuned_baseline_results['TP']}"
)

print("\nTuned Class Weighted")
print(
    f"TN: {lr_tuned_weighted_results['TN']}   "
    f"FP: {lr_tuned_weighted_results['FP']}"
)
print(
    f"FN: {lr_tuned_weighted_results['FN']}   "
    f"TP: {lr_tuned_weighted_results['TP']}"
)


# ------------------------------------------------------------
# Selected hyperparameters and CV results
# ------------------------------------------------------------

print("\n=== Selected Hyperparameters ===\n")

print("Tuned Baseline")
print(
    f"regParam        : "
    f"{lr_baseline_best_model.getRegParam()}"
)
print(
    f"elasticNetParam : "
    f"{lr_baseline_best_model.getElasticNetParam()}"
)
print(
    f"Best CV PR-AUC  : "
    f"{best_cv_pr_auc:.4f}"
)
print(
    f"CV time         : "
    f"{lr_baseline_cv_time:.2f} seconds"
)

print("\nTuned Class Weighted")
print(
    f"regParam        : "
    f"{lr_weighted_best_model.getRegParam()}"
)
print(
    f"elasticNetParam : "
    f"{lr_weighted_best_model.getElasticNetParam()}"
)
print(
    f"Best CV PR-AUC  : "
    f"{weighted_best_cv_pr_auc:.4f}"
)
print(
    f"CV time         : "
    f"{lr_weighted_cv_time:.2f} seconds"
)

=== Tuned Logistic Regression Test Evaluation ===

Metric                    Baseline      Weighted
------------------------------------------------
Accuracy                    0.9349        0.6458
Churn Precision             0.0000        0.0863
Churn Recall                0.0000        0.4636
Churn F1                    0.0000        0.1455
ROC-AUC                     0.5920        0.5881
PR-AUC                      0.0864        0.0818

=== Confusion Matrix Counts ===

Tuned Baseline
TN: 1581   FP: 0
FN: 110   TP: 0

Tuned Class Weighted
TN: 1041   FP: 540
FN: 59   TP: 51

=== Selected Hyperparameters ===

Tuned Baseline
regParam        : 0.01
elasticNetParam : 0.5
Best CV PR-AUC  : 0.0848
CV time         : 19.13 seconds

Tuned Class Weighted
regParam        : 0.1
elasticNetParam : 0.5
Best CV PR-AUC  : 0.0857
CV time         : 23.51 seconds


### Tuned Logistic Regression Interpretation

Following cross-validation, the tuned untreated Logistic Regression model continued to classify every test observation as non-churn. Although this produced an accuracy of 93.49%, churn precision, recall and F1-score remained zero, confirming that the high accuracy resulted from the dominance of the non-churn class rather than effective churn identification.

The tuned class-weighted model identified 51 of the 110 churn customers, producing churn recall of 46.36% and an F1-score of 0.1455. This improvement in churn detection was accompanied by 540 false-positive predictions and consequently low churn precision of 8.63%. Overall accuracy decreased to 64.58%, illustrating the trade-off between majority-class accuracy and minority-class detection.

Compared with the untuned class-weighted model, hyperparameter optimisation produced a more conservative classification outcome, reducing false positives from 629 to 540 while also reducing true-positive churn detections from 57 to 51. Churn precision and F1-score increased slightly, whereas churn recall decreased.

The tuned models achieved ROC-AUC values of 0.5920 and 0.5881 and PR-AUC values of 0.0864 and 0.0818 for the untreated and class-weighted models respectively. These results suggest that although class weighting substantially altered the classification decisions made at the default threshold, Logistic Regression demonstrated limited overall discriminatory performance on this dataset.

# 4. Decision Tree

Decision Tree classification is evaluated as a non-linear alternative to Logistic Regression. Unlike Logistic Regression, Decision Trees can capture non-linear relationships and interactions between predictors without requiring numerical feature scaling.

Both untreated and class-weighted Decision Tree models are optimised using cross-validation on the training data. The unscaled feature representation generated during preprocessing is used, and the held-out test set remains excluded from model fitting and hyperparameter selection.

## 4.1 Baseline Decision Tree Cross-Validation

The untreated Decision Tree model is optimised using five-fold cross-validation. A compact hyperparameter grid evaluates alternative tree depths and minimum node-size requirements, following the parameter-tuning structure used in the taught methodology.

PR-AUC is retained as the optimisation criterion to maintain a consistent evaluation approach across models and to account for the minority churn class. Hyperparameter selection is performed exclusively using the training data.

In [ ]:
# ============================================================
# Cell 9 — Baseline Decision Tree Cross-Validation
# ============================================================

from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ------------------------------------------------------------
# Decision Tree estimator
# ------------------------------------------------------------

dt_cv_baseline = DecisionTreeClassifier(
    featuresCol="features_unscaled",
    labelCol="label",
    seed=42
)

# ------------------------------------------------------------
# Compact hyperparameter grid
# ------------------------------------------------------------

dt_param_grid = (
    ParamGridBuilder()
    .addGrid(
        dt_cv_baseline.maxDepth,
        [5, 10, 15]
    )
    .addGrid(
        dt_cv_baseline.minInstancesPerNode,
        [1, 5]
    )
    .build()
)

# ------------------------------------------------------------
# Five-fold cross-validation
# PR-AUC optimisation
# ------------------------------------------------------------

dt_baseline_cv = CrossValidator(
    estimator=dt_cv_baseline,
    estimatorParamMaps=dt_param_grid,
    evaluator=pr_evaluator,
    numFolds=5,
    seed=42,
    parallelism=2
)

print("=== Baseline Decision Tree CV ===")
print(f"Parameter combinations : {len(dt_param_grid)}")
print("Cross-validation folds : 5")
print(
    f"Total model fits       : "
    f"{len(dt_param_grid) * 5}"
)
print("Optimisation metric    : PR-AUC")
print("\nStarting cross-validation...\n")

# ------------------------------------------------------------
# Fit using TRAINING DATA ONLY
# ------------------------------------------------------------

start_time = time.time()

dt_baseline_cv_model = dt_baseline_cv.fit(
    train_tree
)

dt_baseline_cv_time = time.time() - start_time

# ------------------------------------------------------------
# Extract best model and parameters
# ------------------------------------------------------------

dt_baseline_best_model = (
    dt_baseline_cv_model.bestModel
)

dt_best_depth = (
    dt_baseline_best_model.getMaxDepth()
)

dt_best_min_instances = (
    dt_baseline_best_model.getMinInstancesPerNode()
)

dt_best_cv_pr_auc = max(
    dt_baseline_cv_model.avgMetrics
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "=== Baseline Decision Tree "
    "Cross-Validation Complete ===\n"
)

print(
    f"Best maxDepth             : "
    f"{dt_best_depth}"
)

print(
    f"Best minInstancesPerNode  : "
    f"{dt_best_min_instances}"
)

print(
    f"Best mean CV PR-AUC       : "
    f"{dt_best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time     : "
    f"{dt_baseline_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for params, metric in zip(
    dt_baseline_cv_model.getEstimatorParamMaps(),
    dt_baseline_cv_model.avgMetrics
):
    print(
        f"maxDepth="
        f"{params[dt_cv_baseline.maxDepth]:<2} | "
        f"minInstances="
        f"{params[dt_cv_baseline.minInstancesPerNode]:<2} | "
        f"Mean CV PR-AUC={metric:.4f}"
    )

=== Baseline Decision Tree CV ===
Parameter combinations : 6
Cross-validation folds : 5
Total model fits       : 30
Optimisation metric    : PR-AUC

Starting cross-validation...

=== Baseline Decision Tree Cross-Validation Complete ===

Best maxDepth             : 5
Best minInstancesPerNode  : 1
Best mean CV PR-AUC       : 0.0695
Cross-validation time     : 23.15 seconds

All parameter combinations:
maxDepth=5  | minInstances=1  | Mean CV PR-AUC=0.0695
maxDepth=5  | minInstances=5  | Mean CV PR-AUC=0.0671
maxDepth=10 | minInstances=1  | Mean CV PR-AUC=0.0588
maxDepth=10 | minInstances=5  | Mean CV PR-AUC=0.0600
maxDepth=15 | minInstances=1  | Mean CV PR-AUC=0.0614
maxDepth=15 | minInstances=5  | Mean CV PR-AUC=0.0616


## 4.2 Class-Weighted Decision Tree Cross-Validation

The class-weighted Decision Tree is optimised using the same hyperparameter grid, five-fold cross-validation procedure and PR-AUC optimisation criterion as the untreated Decision Tree. Maintaining identical tuning conditions enables the effect of class weighting to be assessed independently of changes to the model-selection procedure.

The previously calculated class weights are applied during model fitting using the weighted training dataset. The held-out test set remains excluded from cross-validation and hyperparameter selection.

In [ ]:
# ============================================================
# Cell 10 — Class-Weighted Decision Tree Cross-Validation
# ============================================================

# ------------------------------------------------------------
# Weighted Decision Tree estimator
# ------------------------------------------------------------

dt_cv_weighted = DecisionTreeClassifier(
    featuresCol="features_unscaled",
    labelCol="label",
    weightCol="classWeight",
    seed=42
)

# ------------------------------------------------------------
# Same compact hyperparameter grid
# ------------------------------------------------------------

dt_weighted_param_grid = (
    ParamGridBuilder()
    .addGrid(
        dt_cv_weighted.maxDepth,
        [5, 10, 15]
    )
    .addGrid(
        dt_cv_weighted.minInstancesPerNode,
        [1, 5]
    )
    .build()
)

# ------------------------------------------------------------
# Five-fold cross-validation
# Same PR-AUC optimisation criterion
# ------------------------------------------------------------

dt_weighted_cv = CrossValidator(
    estimator=dt_cv_weighted,
    estimatorParamMaps=dt_weighted_param_grid,
    evaluator=pr_evaluator,
    numFolds=5,
    seed=42,
    parallelism=2
)

print("=== Class-Weighted Decision Tree CV ===")
print(f"Parameter combinations : {len(dt_weighted_param_grid)}")
print("Cross-validation folds : 5")
print(
    f"Total model fits       : "
    f"{len(dt_weighted_param_grid) * 5}"
)
print("Optimisation metric    : PR-AUC")
print("\nStarting cross-validation...\n")

# ------------------------------------------------------------
# Fit using WEIGHTED TRAINING DATA ONLY
# ------------------------------------------------------------

start_time = time.time()

dt_weighted_cv_model = dt_weighted_cv.fit(
    train_tree_weighted
)

dt_weighted_cv_time = time.time() - start_time

# ------------------------------------------------------------
# Extract best model and parameters
# ------------------------------------------------------------

dt_weighted_best_model = (
    dt_weighted_cv_model.bestModel
)

dt_weighted_best_depth = (
    dt_weighted_best_model.getMaxDepth()
)

dt_weighted_best_min_instances = (
    dt_weighted_best_model.getMinInstancesPerNode()
)

dt_weighted_best_cv_pr_auc = max(
    dt_weighted_cv_model.avgMetrics
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "=== Class-Weighted Decision Tree "
    "Cross-Validation Complete ===\n"
)

print(
    f"Best maxDepth             : "
    f"{dt_weighted_best_depth}"
)

print(
    f"Best minInstancesPerNode  : "
    f"{dt_weighted_best_min_instances}"
)

print(
    f"Best mean CV PR-AUC       : "
    f"{dt_weighted_best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time     : "
    f"{dt_weighted_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for params, metric in zip(
    dt_weighted_cv_model.getEstimatorParamMaps(),
    dt_weighted_cv_model.avgMetrics
):
    print(
        f"maxDepth="
        f"{params[dt_cv_weighted.maxDepth]:<2} | "
        f"minInstances="
        f"{params[dt_cv_weighted.minInstancesPerNode]:<2} | "
        f"Mean CV PR-AUC={metric:.4f}"
    )

=== Class-Weighted Decision Tree CV ===
Parameter combinations : 6
Cross-validation folds : 5
Total model fits       : 30
Optimisation metric    : PR-AUC

Starting cross-validation...

=== Class-Weighted Decision Tree Cross-Validation Complete ===

Best maxDepth             : 10
Best minInstancesPerNode  : 1
Best mean CV PR-AUC       : 0.0697
Cross-validation time     : 19.84 seconds

All parameter combinations:
maxDepth=5  | minInstances=1  | Mean CV PR-AUC=0.0652
maxDepth=5  | minInstances=5  | Mean CV PR-AUC=0.0652
maxDepth=10 | minInstances=1  | Mean CV PR-AUC=0.0697
maxDepth=10 | minInstances=5  | Mean CV PR-AUC=0.0697
maxDepth=15 | minInstances=1  | Mean CV PR-AUC=0.0613
maxDepth=15 | minInstances=5  | Mean CV PR-AUC=0.0645


### Decision Tree Cross-Validation Interpretation

Cross-validation selected different tree complexities for the untreated and class-weighted Decision Tree models. The untreated model achieved its highest mean cross-validation PR-AUC of 0.0695 using maxDepth = 5 and minInstancesPerNode = 1, while the class-weighted model achieved a marginally higher mean PR-AUC of 0.0697 using maxDepth = 10 and minInstancesPerNode = 1.

The difference in optimal cross-validation PR-AUC was very small, providing no evidence of a substantial improvement in ranking performance from class weighting. However, weighting altered the preferred tree complexity, with the weighted model selecting a deeper tree. Deeper configurations did not consistently improve performance, particularly at maxDepth = 15, indicating that additional model complexity did not necessarily improve generalisation.

## 4.3 Tuned Decision Tree Evaluation

Following hyperparameter selection using five-fold cross-validation, the optimal untreated and class-weighted Decision Tree models are evaluated on the naturally distributed held-out test set. No further hyperparameter changes are made following this evaluation.

Performance is assessed using the common evaluation framework to compare overall predictive performance and, specifically, the ability of each model to identify churn customers.

In [ ]:
# ============================================================
# Cell 11 — Tuned Decision Tree Test Evaluation
# ============================================================

# ------------------------------------------------------------
# Generate held-out test predictions
# ------------------------------------------------------------

dt_tuned_baseline_predictions = (
    dt_baseline_best_model.transform(test_tree)
)

dt_tuned_weighted_predictions = (
    dt_weighted_best_model.transform(test_tree)
)


# ------------------------------------------------------------
# Evaluate both tuned models
# ------------------------------------------------------------

dt_tuned_baseline_results = evaluate_binary_model(
    dt_tuned_baseline_predictions,
    "Decision Tree — Tuned Baseline"
)

dt_tuned_weighted_results = evaluate_binary_model(
    dt_tuned_weighted_predictions,
    "Decision Tree — Tuned Class Weighted"
)


# ------------------------------------------------------------
# Retain CV information
# ------------------------------------------------------------

dt_tuned_baseline_results["CV Time (s)"] = (
    dt_baseline_cv_time
)

dt_tuned_weighted_results["CV Time (s)"] = (
    dt_weighted_cv_time
)

dt_tuned_baseline_results["Best CV PR-AUC"] = (
    dt_best_cv_pr_auc
)

dt_tuned_weighted_results["Best CV PR-AUC"] = (
    dt_weighted_best_cv_pr_auc
)


# ------------------------------------------------------------
# Display tuned model comparison
# ------------------------------------------------------------

print("=== Tuned Decision Tree Test Evaluation ===\n")

print(
    f"{'Metric':<20}"
    f"{'Baseline':>14}"
    f"{'Weighted':>14}"
)

print("-" * 48)

for metric in [
    "Accuracy",
    "Churn Precision",
    "Churn Recall",
    "Churn F1",
    "ROC-AUC",
    "PR-AUC"
]:
    print(
        f"{metric:<20}"
        f"{dt_tuned_baseline_results[metric]:>14.4f}"
        f"{dt_tuned_weighted_results[metric]:>14.4f}"
    )


# ------------------------------------------------------------
# Confusion matrices
# ------------------------------------------------------------

print("\n=== Confusion Matrix Counts ===\n")

print("Tuned Baseline")
print(
    f"TN: {dt_tuned_baseline_results['TN']}   "
    f"FP: {dt_tuned_baseline_results['FP']}"
)
print(
    f"FN: {dt_tuned_baseline_results['FN']}   "
    f"TP: {dt_tuned_baseline_results['TP']}"
)

print("\nTuned Class Weighted")
print(
    f"TN: {dt_tuned_weighted_results['TN']}   "
    f"FP: {dt_tuned_weighted_results['FP']}"
)
print(
    f"FN: {dt_tuned_weighted_results['FN']}   "
    f"TP: {dt_tuned_weighted_results['TP']}"
)


# ------------------------------------------------------------
# Selected hyperparameters and CV results
# ------------------------------------------------------------

print("\n=== Selected Hyperparameters ===\n")

print("Tuned Baseline")
print(
    f"maxDepth            : "
    f"{dt_baseline_best_model.getMaxDepth()}"
)
print(
    f"minInstancesPerNode : "
    f"{dt_baseline_best_model.getMinInstancesPerNode()}"
)
print(
    f"Best CV PR-AUC      : "
    f"{dt_best_cv_pr_auc:.4f}"
)
print(
    f"CV time             : "
    f"{dt_baseline_cv_time:.2f} seconds"
)

print("\nTuned Class Weighted")
print(
    f"maxDepth            : "
    f"{dt_weighted_best_model.getMaxDepth()}"
)
print(
    f"minInstancesPerNode : "
    f"{dt_weighted_best_model.getMinInstancesPerNode()}"
)
print(
    f"Best CV PR-AUC      : "
    f"{dt_weighted_best_cv_pr_auc:.4f}"
)
print(
    f"CV time             : "
    f"{dt_weighted_cv_time:.2f} seconds"
)

=== Tuned Decision Tree Test Evaluation ===

Metric                    Baseline      Weighted
------------------------------------------------
Accuracy                    0.9344        0.6499
Churn Precision             0.0000        0.0727
Churn Recall                0.0000        0.3727
Churn F1                    0.0000        0.1217
ROC-AUC                     0.4829        0.5188
PR-AUC                      0.0653        0.0769

=== Confusion Matrix Counts ===

Tuned Baseline
TN: 1580   FP: 1
FN: 110   TP: 0

Tuned Class Weighted
TN: 1058   FP: 523
FN: 69   TP: 41

=== Selected Hyperparameters ===

Tuned Baseline
maxDepth            : 5
minInstancesPerNode : 1
Best CV PR-AUC      : 0.0695
CV time             : 23.15 seconds

Tuned Class Weighted
maxDepth            : 10
minInstancesPerNode : 1
Best CV PR-AUC      : 0.0697
CV time             : 19.84 seconds


### Tuned Decision Tree Interpretation

The untreated Decision Tree achieved 93.44% accuracy but identified none of the 110 churn customers. It correctly classified 1,580 non-churn observations, with only one false positive, while churn precision, recall and F1-score remained zero.

The class-weighted Decision Tree identified 41 churn customers (37.27% recall), but also generated 523 false positives. Consequently, churn precision was 7.27%, F1-score was 0.1217 and accuracy decreased to 64.99%.

Class weighting improved ROC-AUC from 0.4829 to 0.5188 and PR-AUC from 0.0653 to 0.0769. However, discriminatory performance remained limited, with improved churn detection accompanied by a substantial increase in false positives.

# 5. Random Forest

Random Forest classification is evaluated as an ensemble-based extension of the Decision Tree approach. By combining predictions from multiple decision trees, Random Forest can capture non-linear relationships and feature interactions while reducing the instability and overfitting associated with an individual tree.

Both untreated and class-weighted Random Forest models are optimised using cross-validation on the training data. The unscaled feature representation is retained because tree-based models do not require numerical feature scaling. The held-out test set remains excluded from model fitting and hyperparameter selection.

## 5.1 Baseline Random Forest Cross-Validation

The untreated Random Forest model is optimised using five-fold cross-validation. A compact hyperparameter grid evaluates alternative numbers of trees and maximum tree depths, following the parameter-tuning structure used in the taught methodology.

PR-AUC is retained as the optimisation criterion to maintain consistency across the modelling experiments and account for the minority churn class. Hyperparameter selection is conducted exclusively using the training data.

In [ ]:
# ============================================================
# Cell 12 — Baseline Random Forest Cross-Validation
# ============================================================

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ------------------------------------------------------------
# Random Forest estimator
# ------------------------------------------------------------

rf_cv_baseline = RandomForestClassifier(
    featuresCol="features_unscaled",
    labelCol="label",
    seed=42
)

# ------------------------------------------------------------
# Compact hyperparameter grid
# ------------------------------------------------------------

rf_param_grid = (
    ParamGridBuilder()
    .addGrid(
        rf_cv_baseline.numTrees,
        [50, 100]
    )
    .addGrid(
        rf_cv_baseline.maxDepth,
        [5, 10]
    )
    .build()
)

# ------------------------------------------------------------
# Five-fold cross-validation
# PR-AUC optimisation
# ------------------------------------------------------------

rf_baseline_cv = CrossValidator(
    estimator=rf_cv_baseline,
    estimatorParamMaps=rf_param_grid,
    evaluator=pr_evaluator,
    numFolds=5,
    seed=42,
    parallelism=2
)

print("=== Baseline Random Forest CV ===")
print(f"Parameter combinations : {len(rf_param_grid)}")
print("Cross-validation folds : 5")
print(
    f"Total model fits       : "
    f"{len(rf_param_grid) * 5}"
)
print("Optimisation metric    : PR-AUC")
print("\nStarting cross-validation...\n")

# ------------------------------------------------------------
# Fit using TRAINING DATA ONLY
# ------------------------------------------------------------

start_time = time.time()

rf_baseline_cv_model = rf_baseline_cv.fit(
    train_tree
)

rf_baseline_cv_time = time.time() - start_time

# ------------------------------------------------------------
# Extract best model and parameters
# ------------------------------------------------------------

rf_baseline_best_model = (
    rf_baseline_cv_model.bestModel
)

rf_best_num_trees = (
    rf_baseline_best_model.getNumTrees
)

rf_best_depth = (
    rf_baseline_best_model.getMaxDepth()
)

rf_best_cv_pr_auc = max(
    rf_baseline_cv_model.avgMetrics
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "=== Baseline Random Forest "
    "Cross-Validation Complete ===\n"
)

print(
    f"Best numTrees        : "
    f"{rf_best_num_trees}"
)

print(
    f"Best maxDepth        : "
    f"{rf_best_depth}"
)

print(
    f"Best mean CV PR-AUC  : "
    f"{rf_best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time: "
    f"{rf_baseline_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for params, metric in zip(
    rf_baseline_cv_model.getEstimatorParamMaps(),
    rf_baseline_cv_model.avgMetrics
):
    print(
        f"numTrees="
        f"{params[rf_cv_baseline.numTrees]:<3} | "
        f"maxDepth="
        f"{params[rf_cv_baseline.maxDepth]:<2} | "
        f"Mean CV PR-AUC={metric:.4f}"
    )

=== Baseline Random Forest CV ===
Parameter combinations : 4
Cross-validation folds : 5
Total model fits       : 20
Optimisation metric    : PR-AUC

Starting cross-validation...

=== Baseline Random Forest Cross-Validation Complete ===

Best numTrees        : 100
Best maxDepth        : 5
Best mean CV PR-AUC  : 0.0869
Cross-validation time: 50.47 seconds

All parameter combinations:
numTrees=50  | maxDepth=5  | Mean CV PR-AUC=0.0852
numTrees=50  | maxDepth=10 | Mean CV PR-AUC=0.0840
numTrees=100 | maxDepth=5  | Mean CV PR-AUC=0.0869
numTrees=100 | maxDepth=10 | Mean CV PR-AUC=0.0832


## 5.2 Class-Weighted Random Forest Cross-Validation

The class-weighted Random Forest model is optimised using the same hyperparameter grid, five-fold cross-validation procedure and PR-AUC optimisation criterion as the untreated Random Forest. Maintaining identical tuning conditions allows the effect of class weighting to be assessed independently of changes to the model-selection procedure.

The previously calculated class weights are incorporated during model fitting using the weighted training dataset. The held-out test set remains excluded from cross-validation and hyperparameter selection.

In [ ]:
# ============================================================
# Cell 13 — Class-Weighted Random Forest Cross-Validation
# ============================================================

# ------------------------------------------------------------
# Weighted Random Forest estimator
# ------------------------------------------------------------

rf_cv_weighted = RandomForestClassifier(
    featuresCol="features_unscaled",
    labelCol="label",
    weightCol="classWeight",
    seed=42
)

# ------------------------------------------------------------
# Same compact hyperparameter grid
# ------------------------------------------------------------

rf_weighted_param_grid = (
    ParamGridBuilder()
    .addGrid(
        rf_cv_weighted.numTrees,
        [50, 100]
    )
    .addGrid(
        rf_cv_weighted.maxDepth,
        [5, 10]
    )
    .build()
)

# ------------------------------------------------------------
# Five-fold cross-validation
# Same PR-AUC optimisation criterion
# ------------------------------------------------------------

rf_weighted_cv = CrossValidator(
    estimator=rf_cv_weighted,
    estimatorParamMaps=rf_weighted_param_grid,
    evaluator=pr_evaluator,
    numFolds=5,
    seed=42,
    parallelism=2
)

print("=== Class-Weighted Random Forest CV ===")
print(f"Parameter combinations : {len(rf_weighted_param_grid)}")
print("Cross-validation folds : 5")
print(
    f"Total model fits       : "
    f"{len(rf_weighted_param_grid) * 5}"
)
print("Optimisation metric    : PR-AUC")
print("\nStarting cross-validation...\n")

# ------------------------------------------------------------
# Fit using WEIGHTED TRAINING DATA ONLY
# ------------------------------------------------------------

start_time = time.time()

rf_weighted_cv_model = rf_weighted_cv.fit(
    train_tree_weighted
)

rf_weighted_cv_time = time.time() - start_time

# ------------------------------------------------------------
# Extract best model and parameters
# ------------------------------------------------------------

rf_weighted_best_model = (
    rf_weighted_cv_model.bestModel
)

rf_weighted_best_num_trees = (
    rf_weighted_best_model.getNumTrees
)

rf_weighted_best_depth = (
    rf_weighted_best_model.getMaxDepth()
)

rf_weighted_best_cv_pr_auc = max(
    rf_weighted_cv_model.avgMetrics
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "=== Class-Weighted Random Forest "
    "Cross-Validation Complete ===\n"
)

print(
    f"Best numTrees        : "
    f"{rf_weighted_best_num_trees}"
)

print(
    f"Best maxDepth        : "
    f"{rf_weighted_best_depth}"
)

print(
    f"Best mean CV PR-AUC  : "
    f"{rf_weighted_best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time: "
    f"{rf_weighted_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for params, metric in zip(
    rf_weighted_cv_model.getEstimatorParamMaps(),
    rf_weighted_cv_model.avgMetrics
):
    print(
        f"numTrees="
        f"{params[rf_cv_weighted.numTrees]:<3} | "
        f"maxDepth="
        f"{params[rf_cv_weighted.maxDepth]:<2} | "
        f"Mean CV PR-AUC={metric:.4f}"
    )

=== Class-Weighted Random Forest CV ===
Parameter combinations : 4
Cross-validation folds : 5
Total model fits       : 20
Optimisation metric    : PR-AUC

Starting cross-validation...

=== Class-Weighted Random Forest Cross-Validation Complete ===

Best numTrees        : 50
Best maxDepth        : 5
Best mean CV PR-AUC  : 0.0929
Cross-validation time: 48.77 seconds

All parameter combinations:
numTrees=50  | maxDepth=5  | Mean CV PR-AUC=0.0929
numTrees=50  | maxDepth=10 | Mean CV PR-AUC=0.0766
numTrees=100 | maxDepth=5  | Mean CV PR-AUC=0.0880
numTrees=100 | maxDepth=10 | Mean CV PR-AUC=0.0790


### Random Forest Cross-Validation Interpretation

Cross-validation selected different ensemble sizes for the untreated and class-weighted Random Forest models. The untreated model achieved its highest mean cross-validation PR-AUC of 0.0869 using 100 trees with maxDepth = 5, whereas the class-weighted model achieved a higher mean PR-AUC of 0.0929 using 50 trees with the same maximum depth.

Class weighting therefore produced an improvement in cross-validation PR-AUC for Random Forest, although the magnitude of the improvement remained modest. Both models favoured the shallower tree depth of 5, while increasing tree depth to 10 reduced cross-validation performance across all evaluated configurations. The weighted model also selected the smaller 50-tree ensemble, indicating that increased ensemble size did not necessarily improve generalisation.

## 5.3 Tuned Random Forest Evaluation

Following hyperparameter selection through five-fold cross-validation, the optimal untreated and class-weighted Random Forest models are evaluated on the naturally distributed held-out test set. No further hyperparameter changes are made following this evaluation.

Both models are assessed using the common evaluation framework, enabling comparison of overall performance, minority-class churn detection and ranking performance under identical test conditions.

In [ ]:
# ============================================================
# Cell 14 — Tuned Random Forest Test Evaluation
# ============================================================

# ------------------------------------------------------------
# Generate held-out test predictions
# ------------------------------------------------------------

rf_tuned_baseline_predictions = (
    rf_baseline_best_model.transform(test_tree)
)

rf_tuned_weighted_predictions = (
    rf_weighted_best_model.transform(test_tree)
)


# ------------------------------------------------------------
# Evaluate both tuned models
# ------------------------------------------------------------

rf_tuned_baseline_results = evaluate_binary_model(
    rf_tuned_baseline_predictions,
    "Random Forest — Tuned Baseline"
)

rf_tuned_weighted_results = evaluate_binary_model(
    rf_tuned_weighted_predictions,
    "Random Forest — Tuned Class Weighted"
)


# ------------------------------------------------------------
# Retain CV information
# ------------------------------------------------------------

rf_tuned_baseline_results["CV Time (s)"] = (
    rf_baseline_cv_time
)

rf_tuned_weighted_results["CV Time (s)"] = (
    rf_weighted_cv_time
)

rf_tuned_baseline_results["Best CV PR-AUC"] = (
    rf_best_cv_pr_auc
)

rf_tuned_weighted_results["Best CV PR-AUC"] = (
    rf_weighted_best_cv_pr_auc
)


# ------------------------------------------------------------
# Display tuned model comparison
# ------------------------------------------------------------

print("=== Tuned Random Forest Test Evaluation ===\n")

print(
    f"{'Metric':<20}"
    f"{'Baseline':>14}"
    f"{'Weighted':>14}"
)

print("-" * 48)

for metric in [
    "Accuracy",
    "Churn Precision",
    "Churn Recall",
    "Churn F1",
    "ROC-AUC",
    "PR-AUC"
]:
    print(
        f"{metric:<20}"
        f"{rf_tuned_baseline_results[metric]:>14.4f}"
        f"{rf_tuned_weighted_results[metric]:>14.4f}"
    )


# ------------------------------------------------------------
# Confusion matrices
# ------------------------------------------------------------

print("\n=== Confusion Matrix Counts ===\n")

print("Tuned Baseline")
print(
    f"TN: {rf_tuned_baseline_results['TN']}   "
    f"FP: {rf_tuned_baseline_results['FP']}"
)
print(
    f"FN: {rf_tuned_baseline_results['FN']}   "
    f"TP: {rf_tuned_baseline_results['TP']}"
)

print("\nTuned Class Weighted")
print(
    f"TN: {rf_tuned_weighted_results['TN']}   "
    f"FP: {rf_tuned_weighted_results['FP']}"
)
print(
    f"FN: {rf_tuned_weighted_results['FN']}   "
    f"TP: {rf_tuned_weighted_results['TP']}"
)


# ------------------------------------------------------------
# Selected hyperparameters and CV results
# ------------------------------------------------------------

print("\n=== Selected Hyperparameters ===\n")

print("Tuned Baseline")
print(
    f"numTrees       : "
    f"{rf_baseline_best_model.getNumTrees}"
)
print(
    f"maxDepth       : "
    f"{rf_baseline_best_model.getMaxDepth()}"
)
print(
    f"Best CV PR-AUC : "
    f"{rf_best_cv_pr_auc:.4f}"
)
print(
    f"CV time        : "
    f"{rf_baseline_cv_time:.2f} seconds"
)

print("\nTuned Class Weighted")
print(
    f"numTrees       : "
    f"{rf_weighted_best_model.getNumTrees}"
)
print(
    f"maxDepth       : "
    f"{rf_weighted_best_model.getMaxDepth()}"
)
print(
    f"Best CV PR-AUC : "
    f"{rf_weighted_best_cv_pr_auc:.4f}"
)
print(
    f"CV time        : "
    f"{rf_weighted_cv_time:.2f} seconds"
)

=== Tuned Random Forest Test Evaluation ===

Metric                    Baseline      Weighted
------------------------------------------------
Accuracy                    0.9349        0.5866
Churn Precision             0.0000        0.0787
Churn Recall                0.0000        0.5000
Churn F1                    0.0000        0.1360
ROC-AUC                     0.5920        0.5912
PR-AUC                      0.0879        0.1012

=== Confusion Matrix Counts ===

Tuned Baseline
TN: 1581   FP: 0
FN: 110   TP: 0

Tuned Class Weighted
TN: 937   FP: 644
FN: 55   TP: 55

=== Selected Hyperparameters ===

Tuned Baseline
numTrees       : 100
maxDepth       : 5
Best CV PR-AUC : 0.0869
CV time        : 50.47 seconds

Tuned Class Weighted
numTrees       : 50
maxDepth       : 5
Best CV PR-AUC : 0.0929
CV time        : 48.77 seconds


### Tuned Random Forest Interpretation

The tuned untreated Random Forest achieved an accuracy of 93.49% but classified every held-out observation as non-churn. Consequently, none of the 110 churn customers were identified, resulting in churn precision, recall and F1-score values of zero. As observed with the preceding untreated models, the high accuracy therefore reflects the dominance of the non-churn class rather than effective churn prediction.

The class-weighted Random Forest identified 55 of the 110 churn customers, producing churn recall of 50.00% and an F1-score of 0.1360. This improvement in minority-class detection was accompanied by 644 false-positive predictions, resulting in churn precision of 7.87% and reducing overall accuracy to 58.66%.

The weighted model achieved a test PR-AUC of 0.1012 compared with 0.0879 for the untreated model, consistent with the improvement observed during cross-validation. ROC-AUC remained broadly similar between the two models at approximately 0.59. These results indicate that class weighting improved Random Forest performance with respect to the minority churn class, particularly in terms of recall and PR-AUC, although this was achieved at the cost of a substantial number of false-positive churn predictions.

# 6. Advanced Ensemble Models

Advanced gradient-boosting algorithms are evaluated following the traditional statistical and tree-based classifiers. XGBoost and LightGBM are included as advanced ensemble methods capable of modelling complex non-linear relationships and interactions between customer characteristics.

The same experimental principles are retained throughout this section: model development and hyperparameter selection are conducted using training data only, class imbalance is explicitly considered, and the naturally distributed held-out test set is reserved for final evaluation.

## 6.1 XGBoost

XGBoost is evaluated as an advanced gradient-boosting ensemble method. Unlike Random Forest, which constructs multiple trees through bagging and aggregates their predictions, XGBoost builds trees sequentially, with subsequent trees seeking to improve errors made by the existing ensemble.

Before model development, the installed XGBoost environment and PySpark integration are verified to ensure compatibility with the existing Spark-based modelling workflow.

In [ ]:
# ============================================================
# Cell 15 — XGBoost Environment and Spark Compatibility Check
# ============================================================

import xgboost as xgb

print("=== XGBoost Environment Check ===")
print(f"XGBoost version : {xgb.__version__}")
print(f"Spark version   : {spark.version}")

# Check whether the PySpark XGBoost estimator is available
try:
    from xgboost.spark import SparkXGBClassifier

    print("SparkXGBClassifier available : Yes")
    print("XGBoost Spark integration    : Ready")

except Exception as e:
    print("SparkXGBClassifier available : No")
    print(f"Reason: {type(e).__name__}: {e}")

=== XGBoost Environment Check ===
XGBoost version : 3.4.1
Spark version   : 4.0.4
SparkXGBClassifier available : Yes
XGBoost Spark integration    : Ready


### 6.1.1 XGBoost Configuration and Imbalance Handling

Two XGBoost configurations are considered. The untreated configuration provides a baseline without explicit adjustment for class imbalance. The imbalance-aware configuration uses scale_pos_weight, calculated from the ratio of non-churn to churn observations in the training data, to increase the influence of minority-class observations during optimisation.

The class-imbalance adjustment is derived exclusively from the training partition. The held-out test set is not used to determine class weights, hyperparameters or model configuration.

In [ ]:
# ============================================================
# Cell 16 — XGBoost Configuration and Parameter Check
# ============================================================

from xgboost.spark import SparkXGBClassifier

# ------------------------------------------------------------
# Training-set class imbalance
# ------------------------------------------------------------

train_non_churn = 6323
train_churn = 439

xgb_scale_pos_weight = (
    train_non_churn / train_churn
)

print("=== XGBoost Configuration ===")
print(f"Training non-churn      : {train_non_churn}")
print(f"Training churn          : {train_churn}")
print(
    f"scale_pos_weight        : "
    f"{xgb_scale_pos_weight:.4f}"
)

# ------------------------------------------------------------
# Create untreated estimator
# ------------------------------------------------------------

xgb_baseline = SparkXGBClassifier(
    features_col="features_unscaled",
    label_col="label",
    prediction_col="prediction",
    probability_col="probability",
    raw_prediction_col="rawPrediction",
    seed=42,
    num_workers=1
)

# ------------------------------------------------------------
# Create imbalance-aware estimator
# ------------------------------------------------------------

xgb_weighted = SparkXGBClassifier(
    features_col="features_unscaled",
    label_col="label",
    prediction_col="prediction",
    probability_col="probability",
    raw_prediction_col="rawPrediction",
    seed=42,
    num_workers=1,
    scale_pos_weight=xgb_scale_pos_weight
)

print("\nEstimator creation       : Successful")
print(
    "Baseline scale weighting: None"
)
print(
    f"Weighted scale_pos_weight: "
    f"{xgb_scale_pos_weight:.4f}"
)

print("\nXGBoost estimators ready.")

=== XGBoost Configuration ===
Training non-churn      : 6323
Training churn          : 439
scale_pos_weight        : 14.4032

Estimator creation       : Successful
Baseline scale weighting: None
Weighted scale_pos_weight: 14.4032

XGBoost estimators ready.


### 6.1.2 Baseline XGBoost Cross-Validation

The untreated XGBoost model is optimised using five-fold cross-validation on the training data. A compact hyperparameter grid evaluates the number of boosting trees and maximum tree depth while holding the remaining parameters constant.

The restricted search space limits unnecessary computational complexity while allowing two influential aspects of model capacity to be evaluated. PR-AUC is retained as the optimisation criterion to maintain consistency with the preceding models and to prioritise performance on the imbalanced churn classification problem.

The held-out test set remains excluded from hyperparameter optimisation.

In [ ]:
# ============================================================
# Cell 17 — Baseline XGBoost Cross-Validation
# ============================================================

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ------------------------------------------------------------
# Recreate baseline XGBoost estimator with fixed settings
# ------------------------------------------------------------

xgb_cv_baseline = SparkXGBClassifier(
    features_col="features_unscaled",
    label_col="label",
    prediction_col="prediction",
    probability_col="probability",
    raw_prediction_col="rawPrediction",

    seed=42,
    num_workers=1,

    # Fixed boosting settings
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0
)

# ------------------------------------------------------------
# Access parameters explicitly
# This avoids relying on version-specific attribute behaviour
# ------------------------------------------------------------

xgb_n_estimators_param = (
    xgb_cv_baseline.getParam("n_estimators")
)

xgb_max_depth_param = (
    xgb_cv_baseline.getParam("max_depth")
)

# ------------------------------------------------------------
# Compact hyperparameter grid
# ------------------------------------------------------------

xgb_baseline_param_grid = (
    ParamGridBuilder()
    .addGrid(
        xgb_n_estimators_param,
        [50, 100]
    )
    .addGrid(
        xgb_max_depth_param,
        [3, 6]
    )
    .build()
)

# ------------------------------------------------------------
# Five-fold cross-validation
# PR-AUC optimisation
# ------------------------------------------------------------

xgb_baseline_cv = CrossValidator(
    estimator=xgb_cv_baseline,
    estimatorParamMaps=xgb_baseline_param_grid,
    evaluator=pr_evaluator,
    numFolds=5,
    seed=42,
    parallelism=2
)

print("=== Baseline XGBoost CV ===")
print(
    f"Parameter combinations : "
    f"{len(xgb_baseline_param_grid)}"
)
print("Cross-validation folds : 5")
print(
    f"Total model fits       : "
    f"{len(xgb_baseline_param_grid) * 5}"
)
print("Optimisation metric    : PR-AUC")
print("learning_rate fixed    : 0.1")

print("\nStarting cross-validation...\n")

# ------------------------------------------------------------
# Fit using TRAINING DATA ONLY
# ------------------------------------------------------------

start_time = time.time()

xgb_baseline_cv_model = (
    xgb_baseline_cv.fit(train_tree)
)

xgb_baseline_cv_time = (
    time.time() - start_time
)

# ------------------------------------------------------------
# Extract best result
# ------------------------------------------------------------

xgb_baseline_metrics = (
    xgb_baseline_cv_model.avgMetrics
)

xgb_baseline_param_maps = (
    xgb_baseline_cv_model.getEstimatorParamMaps()
)

xgb_baseline_best_index = max(
    range(len(xgb_baseline_metrics)),
    key=lambda i: xgb_baseline_metrics[i]
)

xgb_baseline_best_params = (
    xgb_baseline_param_maps[
        xgb_baseline_best_index
    ]
)

xgb_baseline_best_model = (
    xgb_baseline_cv_model.bestModel
)

xgb_baseline_best_n_estimators = (
    xgb_baseline_best_params[
        xgb_n_estimators_param
    ]
)

xgb_baseline_best_max_depth = (
    xgb_baseline_best_params[
        xgb_max_depth_param
    ]
)

xgb_baseline_best_cv_pr_auc = (
    xgb_baseline_metrics[
        xgb_baseline_best_index
    ]
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "=== Baseline XGBoost "
    "Cross-Validation Complete ===\n"
)

print(
    f"Best n_estimators     : "
    f"{xgb_baseline_best_n_estimators}"
)

print(
    f"Best max_depth        : "
    f"{xgb_baseline_best_max_depth}"
)

print(
    f"Best mean CV PR-AUC   : "
    f"{xgb_baseline_best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time : "
    f"{xgb_baseline_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for params, metric in zip(
    xgb_baseline_param_maps,
    xgb_baseline_metrics
):
    print(
        f"n_estimators="
        f"{params[xgb_n_estimators_param]:<3} | "
        f"max_depth="
        f"{params[xgb_max_depth_param]:<2} | "
        f"Mean CV PR-AUC={metric:.4f}"
    )

=== Baseline XGBoost CV ===
Parameter combinations : 4
Cross-validation folds : 5
Total model fits       : 20
Optimisation metric    : PR-AUC
learning_rate fixed    : 0.1

Starting cross-validation...



INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'colsample_bytree': 1.0, 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 6, 'objective': 'binary:logistic', 'subsample': 1.0, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'colsample_bytree': 1.0, 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 3, 'objective': 'binary:logistic', 'subsample': 1.0, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'colsample_bytree': 1.0, 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 3, 'objective': 'binary:logistic', '

=== Baseline XGBoost Cross-Validation Complete ===

Best n_estimators     : 50
Best max_depth        : 3
Best mean CV PR-AUC   : 0.0826
Cross-validation time : 46.31 seconds

All parameter combinations:
n_estimators=50  | max_depth=3  | Mean CV PR-AUC=0.0826
n_estimators=50  | max_depth=6  | Mean CV PR-AUC=0.0775
n_estimators=100 | max_depth=3  | Mean CV PR-AUC=0.0816
n_estimators=100 | max_depth=6  | Mean CV PR-AUC=0.0760


### 6.1.3 Imbalance-Aware XGBoost Cross-Validation

The imbalance-aware XGBoost model is optimised using the same hyperparameter grid, five-fold cross-validation procedure and PR-AUC optimisation criterion as the untreated model. The only experimental change is the application of scale_pos_weight = 14.4032, derived from the ratio of non-churn to churn observations in the training partition.

Maintaining otherwise identical tuning conditions enables the effect of XGBoost's native class-imbalance adjustment to be assessed independently. The held-out test set remains excluded from model fitting and hyperparameter selection.

In [ ]:
# ============================================================
# Cell 18 — Imbalance-Aware XGBoost Cross-Validation
# ============================================================

# ------------------------------------------------------------
# Imbalance-aware XGBoost estimator
# ------------------------------------------------------------

xgb_cv_weighted = SparkXGBClassifier(
    features_col="features_unscaled",
    label_col="label",
    prediction_col="prediction",
    probability_col="probability",
    raw_prediction_col="rawPrediction",

    seed=42,
    num_workers=1,

    # Training-derived imbalance adjustment
    scale_pos_weight=xgb_scale_pos_weight,

    # Same fixed boosting settings as baseline
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0
)

# ------------------------------------------------------------
# Access estimator parameters
# ------------------------------------------------------------

xgb_weighted_n_estimators_param = (
    xgb_cv_weighted.getParam("n_estimators")
)

xgb_weighted_max_depth_param = (
    xgb_cv_weighted.getParam("max_depth")
)

# ------------------------------------------------------------
# Identical compact hyperparameter grid
# ------------------------------------------------------------

xgb_weighted_param_grid = (
    ParamGridBuilder()
    .addGrid(
        xgb_weighted_n_estimators_param,
        [50, 100]
    )
    .addGrid(
        xgb_weighted_max_depth_param,
        [3, 6]
    )
    .build()
)

# ------------------------------------------------------------
# Five-fold cross-validation
# ------------------------------------------------------------

xgb_weighted_cv = CrossValidator(
    estimator=xgb_cv_weighted,
    estimatorParamMaps=xgb_weighted_param_grid,
    evaluator=pr_evaluator,
    numFolds=5,
    seed=42,
    parallelism=2
)

print("=== Imbalance-Aware XGBoost CV ===")
print(
    f"Parameter combinations : "
    f"{len(xgb_weighted_param_grid)}"
)
print("Cross-validation folds : 5")
print(
    f"Total model fits       : "
    f"{len(xgb_weighted_param_grid) * 5}"
)
print("Optimisation metric    : PR-AUC")
print("learning_rate fixed    : 0.1")
print(
    f"scale_pos_weight       : "
    f"{xgb_scale_pos_weight:.4f}"
)

print("\nStarting cross-validation...\n")

# ------------------------------------------------------------
# Fit using TRAINING DATA ONLY
# ------------------------------------------------------------

start_time = time.time()

xgb_weighted_cv_model = (
    xgb_weighted_cv.fit(train_tree)
)

xgb_weighted_cv_time = (
    time.time() - start_time
)

# ------------------------------------------------------------
# Extract best result
# ------------------------------------------------------------

xgb_weighted_metrics = (
    xgb_weighted_cv_model.avgMetrics
)

xgb_weighted_param_maps = (
    xgb_weighted_cv_model.getEstimatorParamMaps()
)

xgb_weighted_best_index = max(
    range(len(xgb_weighted_metrics)),
    key=lambda i: xgb_weighted_metrics[i]
)

xgb_weighted_best_params = (
    xgb_weighted_param_maps[
        xgb_weighted_best_index
    ]
)

xgb_weighted_best_model = (
    xgb_weighted_cv_model.bestModel
)

xgb_weighted_best_n_estimators = (
    xgb_weighted_best_params[
        xgb_weighted_n_estimators_param
    ]
)

xgb_weighted_best_max_depth = (
    xgb_weighted_best_params[
        xgb_weighted_max_depth_param
    ]
)

xgb_weighted_best_cv_pr_auc = (
    xgb_weighted_metrics[
        xgb_weighted_best_index
    ]
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "=== Imbalance-Aware XGBoost "
    "Cross-Validation Complete ===\n"
)

print(
    f"Best n_estimators     : "
    f"{xgb_weighted_best_n_estimators}"
)

print(
    f"Best max_depth        : "
    f"{xgb_weighted_best_max_depth}"
)

print(
    f"Best mean CV PR-AUC   : "
    f"{xgb_weighted_best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time : "
    f"{xgb_weighted_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for params, metric in zip(
    xgb_weighted_param_maps,
    xgb_weighted_metrics
):
    print(
        f"n_estimators="
        f"{params[xgb_weighted_n_estimators_param]:<3} | "
        f"max_depth="
        f"{params[xgb_weighted_max_depth_param]:<2} | "
        f"Mean CV PR-AUC={metric:.4f}"
    )

=== Imbalance-Aware XGBoost CV ===
Parameter combinations : 4
Cross-validation folds : 5
Total model fits       : 20
Optimisation metric    : PR-AUC
learning_rate fixed    : 0.1
scale_pos_weight       : 14.4032

Starting cross-validation...



INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'colsample_bytree': 1.0, 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 3, 'objective': 'binary:logistic', 'scale_pos_weight': 14.403189066059225, 'subsample': 1.0, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'colsample_bytree': 1.0, 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 6, 'objective': 'binary:logistic', 'scale_pos_weight': 14.403189066059225, 'subsample': 1.0, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'colsample_bytree': 1.0, 'device'

=== Imbalance-Aware XGBoost Cross-Validation Complete ===

Best n_estimators     : 50
Best max_depth        : 3
Best mean CV PR-AUC   : 0.0785
Cross-validation time : 41.02 seconds

All parameter combinations:
n_estimators=50  | max_depth=3  | Mean CV PR-AUC=0.0785
n_estimators=50  | max_depth=6  | Mean CV PR-AUC=0.0697
n_estimators=100 | max_depth=3  | Mean CV PR-AUC=0.0777
n_estimators=100 | max_depth=6  | Mean CV PR-AUC=0.0705


### XGBoost Cross-Validation Interpretation

Cross-validation selected the same hyperparameter configuration for both untreated and imbalance-aware XGBoost: 50 boosting rounds with max_depth = 3.

The untreated XGBoost model achieved a mean cross-validation PR-AUC of 0.0826, whereas the imbalance-aware configuration achieved a slightly lower value of 0.0785. Therefore, applying scale_pos_weight = 14.4032 did not improve PR-AUC during cross-validation.

Across both configurations, shallower trees and the smaller number of boosting rounds generally performed better than the more complex alternatives. This suggests that increasing XGBoost model complexity did not improve generalisation within the evaluated search space.

Although the imbalance-aware configuration produced lower cross-validation PR-AUC, its effect on minority-class classification performance will be assessed separately using the held-out test set.

### 6.1.4 Tuned XGBoost Evaluation

Following hyperparameter optimisation, the best untreated and imbalance-aware XGBoost models are evaluated on the held-out test set. No further changes to hyperparameters, imbalance treatment or model configuration are made following this evaluation.

The common evaluation framework is used to compare overall classification performance, churn-class precision, recall and F1-score, together with ROC-AUC, PR-AUC and confusion-matrix values.

In [ ]:
# ============================================================
# Cell 19 — Tuned XGBoost Test Evaluation
# ============================================================

# ------------------------------------------------------------
# Generate held-out test predictions
# ------------------------------------------------------------

xgb_tuned_baseline_predictions = (
    xgb_baseline_best_model.transform(test_tree)
)

xgb_tuned_weighted_predictions = (
    xgb_weighted_best_model.transform(test_tree)
)


# ------------------------------------------------------------
# Evaluate both tuned models
# ------------------------------------------------------------

xgb_tuned_baseline_results = evaluate_binary_model(
    xgb_tuned_baseline_predictions,
    "XGBoost — Tuned Baseline"
)

xgb_tuned_weighted_results = evaluate_binary_model(
    xgb_tuned_weighted_predictions,
    "XGBoost — Tuned Imbalance-Aware"
)


# ------------------------------------------------------------
# Retain CV information
# ------------------------------------------------------------

xgb_tuned_baseline_results["CV Time (s)"] = (
    xgb_baseline_cv_time
)

xgb_tuned_weighted_results["CV Time (s)"] = (
    xgb_weighted_cv_time
)

xgb_tuned_baseline_results["Best CV PR-AUC"] = (
    xgb_baseline_best_cv_pr_auc
)

xgb_tuned_weighted_results["Best CV PR-AUC"] = (
    xgb_weighted_best_cv_pr_auc
)


# ------------------------------------------------------------
# Display comparison
# ------------------------------------------------------------

print("=== Tuned XGBoost Test Evaluation ===\n")

print(
    f"{'Metric':<20}"
    f"{'Baseline':>14}"
    f"{'Imbalance-Aware':>18}"
)

print("-" * 52)

for metric in [
    "Accuracy",
    "Churn Precision",
    "Churn Recall",
    "Churn F1",
    "ROC-AUC",
    "PR-AUC"
]:
    print(
        f"{metric:<20}"
        f"{xgb_tuned_baseline_results[metric]:>14.4f}"
        f"{xgb_tuned_weighted_results[metric]:>18.4f}"
    )


# ------------------------------------------------------------
# Confusion matrices
# ------------------------------------------------------------

print("\n=== Confusion Matrix Counts ===\n")

print("Tuned Baseline")
print(
    f"TN: {xgb_tuned_baseline_results['TN']}   "
    f"FP: {xgb_tuned_baseline_results['FP']}"
)
print(
    f"FN: {xgb_tuned_baseline_results['FN']}   "
    f"TP: {xgb_tuned_baseline_results['TP']}"
)

print("\nTuned Imbalance-Aware")
print(
    f"TN: {xgb_tuned_weighted_results['TN']}   "
    f"FP: {xgb_tuned_weighted_results['FP']}"
)
print(
    f"FN: {xgb_tuned_weighted_results['FN']}   "
    f"TP: {xgb_tuned_weighted_results['TP']}"
)


# ------------------------------------------------------------
# Selected hyperparameters
# ------------------------------------------------------------

print("\n=== Selected Hyperparameters ===\n")

print("Tuned Baseline")
print(
    f"n_estimators   : "
    f"{xgb_baseline_best_n_estimators}"
)
print(
    f"max_depth      : "
    f"{xgb_baseline_best_max_depth}"
)
print(
    f"Best CV PR-AUC : "
    f"{xgb_baseline_best_cv_pr_auc:.4f}"
)
print(
    f"CV time        : "
    f"{xgb_baseline_cv_time:.2f} seconds"
)

print("\nTuned Imbalance-Aware")
print(
    f"n_estimators   : "
    f"{xgb_weighted_best_n_estimators}"
)
print(
    f"max_depth      : "
    f"{xgb_weighted_best_max_depth}"
)
print(
    f"scale_pos_weight : "
    f"{xgb_scale_pos_weight:.4f}"
)
print(
    f"Best CV PR-AUC : "
    f"{xgb_weighted_best_cv_pr_auc:.4f}"
)
print(
    f"CV time        : "
    f"{xgb_weighted_cv_time:.2f} seconds"
)

=== Tuned XGBoost Test Evaluation ===

Metric                    Baseline   Imbalance-Aware
----------------------------------------------------
Accuracy                    0.9349            0.5648
Churn Precision             0.0000            0.0770
Churn Recall                0.0000            0.5182
Churn F1                    0.0000            0.1341
ROC-AUC                     0.5886            0.5758
PR-AUC                      0.0823            0.0854

=== Confusion Matrix Counts ===

Tuned Baseline
TN: 1581   FP: 0
FN: 110   TP: 0

Tuned Imbalance-Aware
TN: 898   FP: 683
FN: 53   TP: 57

=== Selected Hyperparameters ===

Tuned Baseline
n_estimators   : 50
max_depth      : 3
Best CV PR-AUC : 0.0826
CV time        : 46.31 seconds

Tuned Imbalance-Aware
n_estimators   : 50
max_depth      : 3
scale_pos_weight : 14.4032
Best CV PR-AUC : 0.0785
CV time        : 41.02 seconds


### Tuned XGBoost Interpretation

The untreated XGBoost model achieved 93.49% accuracy but classified all test observations as non-churn, identifying none of the 110 churn customers.

The imbalance-aware XGBoost model identified 57 churn customers (51.82% recall), alongside 683 false positives. This produced churn precision of 7.70%, an F1-score of 0.1341 and accuracy of 56.48%.

Test PR-AUC increased slightly from 0.0823 to 0.0854, while ROC-AUC decreased from 0.5886 to 0.5758. This contrasts with the small reduction in mean cross-validation PR-AUC, illustrating that minor performance differences can vary across validation samples.

Overall, scale_pos_weight increased sensitivity to churn but also increased false positives, demonstrating a trade-off rather than a consistent improvement across all metrics.

## 6.2 LightGBM

LightGBM is evaluated as a second advanced gradient-boosting ensemble method. The algorithm constructs decision-tree ensembles sequentially and is designed to provide efficient learning for structured classification problems.

LightGBM extends the advanced ensemble comparison alongside XGBoost while retaining the same experimental principles applied throughout the study. Model development and hyperparameter selection are restricted to the training data, class imbalance is explicitly considered, and the held-out test set remains reserved for final evaluation.

Before model development, the available LightGBM environment is examined to determine the most appropriate implementation for the existing modelling workflow.

In [ ]:
# ============================================================
# Cell 20 — LightGBM Environment and Compatibility Check
# ============================================================

import importlib.util

print("=== LightGBM Environment Check ===")
print(f"Spark version : {spark.version}")

# ------------------------------------------------------------
# Check standard Python LightGBM
# ------------------------------------------------------------

lightgbm_available = (
    importlib.util.find_spec("lightgbm") is not None
)

print(
    f"Python LightGBM installed : "
    f"{'Yes' if lightgbm_available else 'No'}"
)

if lightgbm_available:
    import lightgbm as lgb

    print(
        f"LightGBM version          : "
        f"{lgb.__version__}"
    )


# ------------------------------------------------------------
# Check SynapseML Spark LightGBM
# ------------------------------------------------------------

synapseml_available = (
    importlib.util.find_spec("synapse") is not None
)

print(
    f"SynapseML package found   : "
    f"{'Yes' if synapseml_available else 'No'}"
)

try:
    from synapse.ml.lightgbm import LightGBMClassifier

    print("Spark LightGBM classifier : Available")
    print("SynapseML integration     : Ready")

except Exception as e:

    print("Spark LightGBM classifier : Not available")
    print(
        f"Reason                     : "
        f"{type(e).__name__}: {e}"
    )

=== LightGBM Environment Check ===
Spark version : 4.0.4
Python LightGBM installed : Yes
LightGBM version          : 4.6.0
SynapseML package found   : No
Spark LightGBM classifier : Not available
Reason                     : ModuleNotFoundError: No module named 'synapse'


### 6.2.1 LightGBM Implementation Strategy

The standard Python LightGBM implementation is used because the Spark-integrated SynapseML LightGBM estimator is not available in the current execution environment. The prepared modelling datasets remain those generated through the PySpark preprocessing pipeline.

Given the modest dataset size, the preprocessed training and held-out test feature matrices can be converted to local numerical arrays for LightGBM model development without creating a meaningful memory or scalability constraint.

The existing train-test separation is preserved. Hyperparameter selection will be conducted using the training partition only, while the held-out test data remain excluded until final model evaluation.

In [ ]:
# ============================================================
# Cell 21 — Prepare Local LightGBM Matrices
# ============================================================

import numpy as np
import pandas as pd

print("=== Preparing LightGBM Matrices ===")

# ------------------------------------------------------------
# Helper: convert Spark vector column to NumPy matrix
# ------------------------------------------------------------

def spark_vector_df_to_numpy(
    spark_df,
    features_col="features_unscaled",
    label_col="label"
):
    rows = (
        spark_df
        .select(features_col, label_col)
        .collect()
    )

    X = np.vstack([
        row[features_col].toArray()
        for row in rows
    ])

    y = np.array([
        row[label_col]
        for row in rows
    ]).astype(int)

    return X, y


# ------------------------------------------------------------
# Convert existing prepared partitions
# ------------------------------------------------------------

X_train_lgbm, y_train_lgbm = (
    spark_vector_df_to_numpy(train_tree)
)

X_test_lgbm, y_test_lgbm = (
    spark_vector_df_to_numpy(test_tree)
)


# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

print(
    f"Training matrix shape : "
    f"{X_train_lgbm.shape}"
)

print(
    f"Test matrix shape     : "
    f"{X_test_lgbm.shape}"
)

print(
    f"Training labels       : "
    f"{y_train_lgbm.shape}"
)

print(
    f"Test labels           : "
    f"{y_test_lgbm.shape}"
)

print(
    f"Training churn count  : "
    f"{int(y_train_lgbm.sum())}"
)

print(
    f"Test churn count      : "
    f"{int(y_test_lgbm.sum())}"
)

print(
    f"Feature count         : "
    f"{X_train_lgbm.shape[1]}"
)

# ------------------------------------------------------------
# Safety assertions
# ------------------------------------------------------------

assert X_train_lgbm.shape == (6762, 24)
assert X_test_lgbm.shape == (1691, 24)

assert len(y_train_lgbm) == 6762
assert len(y_test_lgbm) == 1691

assert int(y_train_lgbm.sum()) == 439
assert int(y_test_lgbm.sum()) == 110

print("\nLightGBM matrices ready.")
print("Held-out test partition preserved.")

=== Preparing LightGBM Matrices ===
Training matrix shape : (6762, 24)
Test matrix shape     : (1691, 24)
Training labels       : (6762,)
Test labels           : (1691,)
Training churn count  : 439
Test churn count      : 110
Feature count         : 24

LightGBM matrices ready.
Held-out test partition preserved.


### 6.2.2 Baseline LightGBM Cross-Validation

The untreated LightGBM classifier is optimised using five-fold stratified cross-validation on the prepared training partition. Stratification preserves the minority churn proportion within each fold.

A compact hyperparameter grid evaluates the number of boosting iterations and maximum tree depth while holding the learning rate constant. PR-AUC is retained as the optimisation criterion to maintain the imbalance-aware evaluation focus used throughout the modelling experiments.

The held-out test set is not used during model fitting or hyperparameter selection.

In [ ]:
# ============================================================
# Cell 22 — Baseline LightGBM Cross-Validation
# ============================================================

import time
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import precision_recall_curve, auc


# ------------------------------------------------------------
# PR-AUC helper
# ------------------------------------------------------------

def calculate_pr_auc(y_true, y_probability):
    precision, recall, _ = precision_recall_curve(
        y_true,
        y_probability
    )

    return auc(recall, precision)


# ------------------------------------------------------------
# Five-fold stratified CV
# ------------------------------------------------------------

lgbm_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ------------------------------------------------------------
# Compact parameter grid
# ------------------------------------------------------------

lgbm_baseline_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 6]
}

lgbm_baseline_param_grid = list(
    ParameterGrid(lgbm_baseline_grid)
)


print("=== Baseline LightGBM CV ===")
print(
    f"Parameter combinations : "
    f"{len(lgbm_baseline_param_grid)}"
)
print("Cross-validation folds : 5")
print(
    f"Total model fits       : "
    f"{len(lgbm_baseline_param_grid) * 5}"
)
print("Optimisation metric    : PR-AUC")
print("learning_rate fixed    : 0.1")

print("\nStarting cross-validation...\n")


# ------------------------------------------------------------
# Cross-validation
# ------------------------------------------------------------

start_time = time.time()

lgbm_baseline_cv_results = []

for params in lgbm_baseline_param_grid:

    fold_scores = []

    for train_idx, val_idx in lgbm_cv.split(
        X_train_lgbm,
        y_train_lgbm
    ):

        X_fold_train = X_train_lgbm[train_idx]
        X_fold_val = X_train_lgbm[val_idx]

        y_fold_train = y_train_lgbm[train_idx]
        y_fold_val = y_train_lgbm[val_idx]

        model = lgb.LGBMClassifier(
            objective="binary",

            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],

            learning_rate=0.1,

            random_state=42,
            n_jobs=-1,

            verbosity=-1
        )

        model.fit(
            X_fold_train,
            y_fold_train
        )

        val_probability = model.predict_proba(
            X_fold_val
        )[:, 1]

        fold_pr_auc = calculate_pr_auc(
            y_fold_val,
            val_probability
        )

        fold_scores.append(fold_pr_auc)

    mean_pr_auc = float(
        np.mean(fold_scores)
    )

    lgbm_baseline_cv_results.append({
        "params": params,
        "mean_pr_auc": mean_pr_auc,
        "fold_scores": fold_scores
    })


lgbm_baseline_cv_time = (
    time.time() - start_time
)


# ------------------------------------------------------------
# Select best configuration
# ------------------------------------------------------------

lgbm_baseline_best_result = max(
    lgbm_baseline_cv_results,
    key=lambda x: x["mean_pr_auc"]
)

lgbm_baseline_best_params = (
    lgbm_baseline_best_result["params"]
)

lgbm_baseline_best_cv_pr_auc = (
    lgbm_baseline_best_result["mean_pr_auc"]
)


# ------------------------------------------------------------
# Fit best model on FULL TRAINING SET
# ------------------------------------------------------------

lgbm_baseline_best_model = lgb.LGBMClassifier(
    objective="binary",

    n_estimators=(
        lgbm_baseline_best_params["n_estimators"]
    ),

    max_depth=(
        lgbm_baseline_best_params["max_depth"]
    ),

    learning_rate=0.1,

    random_state=42,
    n_jobs=-1,

    verbosity=-1
)

lgbm_baseline_best_model.fit(
    X_train_lgbm,
    y_train_lgbm
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "=== Baseline LightGBM "
    "Cross-Validation Complete ===\n"
)

print(
    f"Best n_estimators     : "
    f"{lgbm_baseline_best_params['n_estimators']}"
)

print(
    f"Best max_depth        : "
    f"{lgbm_baseline_best_params['max_depth']}"
)

print(
    f"Best mean CV PR-AUC   : "
    f"{lgbm_baseline_best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time : "
    f"{lgbm_baseline_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for result in lgbm_baseline_cv_results:

    params = result["params"]

    print(
        f"n_estimators="
        f"{params['n_estimators']:<3} | "
        f"max_depth="
        f"{params['max_depth']:<2} | "
        f"Mean CV PR-AUC="
        f"{result['mean_pr_auc']:.4f}"
    )

=== Baseline LightGBM CV ===
Parameter combinations : 4
Cross-validation folds : 5
Total model fits       : 20
Optimisation metric    : PR-AUC
learning_rate fixed    : 0.1

Starting cross-validation...



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

=== Baseline LightGBM Cross-Validation Complete ===

Best n_estimators     : 100
Best max_depth        : 3
Best mean CV PR-AUC   : 0.0812
Cross-validation time : 2.67 seconds

All parameter combinations:
n_estimators=50  | max_depth=3  | Mean CV PR-AUC=0.0810
n_estimators=100 | max_depth=3  | Mean CV PR-AUC=0.0812
n_estimators=50  | max_depth=6  | Mean CV PR-AUC=0.0747
n_estimators=100 | max_depth=6  | Mean CV PR-AUC=0.0758


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# ============================================================
# Cell 23 — Attach Preserved Feature Names for LightGBM
# ============================================================

import pandas as pd

print("=== LightGBM Feature-Name Validation ===")

# ------------------------------------------------------------
# Validate existing feature-name list
# ------------------------------------------------------------

assert isinstance(feature_names, list)
assert len(feature_names) == 24

print(f"Feature count : {len(feature_names)}")

for i, name in enumerate(feature_names):
    print(f"{i:>2}: {name}")


# ------------------------------------------------------------
# Create named LightGBM DataFrames
# ------------------------------------------------------------

X_train_lgbm_named = pd.DataFrame(
    X_train_lgbm,
    columns=feature_names
)

X_test_lgbm_named = pd.DataFrame(
    X_test_lgbm,
    columns=feature_names
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert X_train_lgbm_named.shape == (6762, 24)
assert X_test_lgbm_named.shape == (1691, 24)

assert not X_train_lgbm_named.isnull().any().any()
assert not X_test_lgbm_named.isnull().any().any()

# Verify values were not altered during conversion
assert (
    X_train_lgbm_named.to_numpy()
    == X_train_lgbm
).all()

assert (
    X_test_lgbm_named.to_numpy()
    == X_test_lgbm
).all()


print("\nNamed LightGBM matrices ready.")
print(
    f"Training shape : {X_train_lgbm_named.shape}"
)
print(
    f"Test shape     : {X_test_lgbm_named.shape}"
)

print("\nFeature names preserved for later explainability.")

=== LightGBM Feature-Name Validation ===
Feature count : 24
 0: CRM_PID_Value_Segment_Bronze
 1: CRM_PID_Value_Segment_Silver
 2: CRM_PID_Value_Segment_Gold
 3: CRM_PID_Value_Segment_Platinum
 4: CRM_PID_Value_Segment_SME
 5: CRM_PID_Value_Segment_Iron
 6: CRM_PID_Value_Segment_SE
 7: CRM_PID_Value_Segment_Lead
 8: CRM_PID_Value_Segment_Unknown
 9: CRM_PID_Value_Segment___unknown
10: EffectiveSegment_SOHO
11: EffectiveSegment_VSE
12: EffectiveSegment_SME
13: EffectiveSegment_SE
14: EffectiveSegment_Other
15: EffectiveSegment_LE
16: EffectiveSegment___unknown
17: Active_subscribers
18: Not_Active_subscribers
19: Suspended_subscribers
20: AvgMobileRevenue
21: AvgFIXRevenue
22: ARPU
23: Inactive_Ratio

Named LightGBM matrices ready.
Training shape : (6762, 24)
Test shape     : (1691, 24)

Feature names preserved for later explainability.


### 6.2.3 Imbalance-Aware LightGBM Cross-Validation

An imbalance-aware LightGBM model is evaluated using the same five-fold stratified cross-validation procedure and hyperparameter grid as the untreated baseline. This maintains a controlled comparison in which the principal difference between the two configurations is the treatment of class imbalance.

The scale_pos_weight parameter is set using the ratio of non-churn to churn observations in the training partition. This increases the contribution of minority-class observations during model training without altering the natural class distribution through under- or oversampling.

PR-AUC remains the optimisation criterion, and the held-out test partition remains excluded from hyperparameter selection.

In [ ]:
# ============================================================
# Cell 24 — Imbalance-Aware LightGBM Cross-Validation
# ============================================================

import time
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold, ParameterGrid


# ------------------------------------------------------------
# Calculate imbalance ratio from TRAINING DATA ONLY
# ------------------------------------------------------------

lgbm_negative_count = int(
    np.sum(y_train_lgbm == 0)
)

lgbm_positive_count = int(
    np.sum(y_train_lgbm == 1)
)

lgbm_scale_pos_weight = (
    lgbm_negative_count / lgbm_positive_count
)


# ------------------------------------------------------------
# Five-fold stratified CV
# ------------------------------------------------------------

lgbm_weighted_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ------------------------------------------------------------
# Same compact grid as baseline
# ------------------------------------------------------------

lgbm_weighted_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 6]
}

lgbm_weighted_param_grid = list(
    ParameterGrid(lgbm_weighted_grid)
)


print("=== Imbalance-Aware LightGBM CV ===")

print(
    f"Training non-churn      : "
    f"{lgbm_negative_count}"
)

print(
    f"Training churn          : "
    f"{lgbm_positive_count}"
)

print(
    f"scale_pos_weight        : "
    f"{lgbm_scale_pos_weight:.4f}"
)

print(
    f"Parameter combinations  : "
    f"{len(lgbm_weighted_param_grid)}"
)

print("Cross-validation folds  : 5")

print(
    f"Total model fits        : "
    f"{len(lgbm_weighted_param_grid) * 5}"
)

print("Optimisation metric     : PR-AUC")
print("learning_rate fixed     : 0.1")

print("\nStarting cross-validation...\n")


# ------------------------------------------------------------
# Cross-validation
# ------------------------------------------------------------

start_time = time.time()

lgbm_weighted_cv_results = []

for params in lgbm_weighted_param_grid:

    fold_scores = []

    for train_idx, val_idx in lgbm_weighted_cv.split(
        X_train_lgbm_named,
        y_train_lgbm
    ):

        X_fold_train = (
            X_train_lgbm_named.iloc[train_idx]
        )

        X_fold_val = (
            X_train_lgbm_named.iloc[val_idx]
        )

        y_fold_train = y_train_lgbm[train_idx]
        y_fold_val = y_train_lgbm[val_idx]

        model = lgb.LGBMClassifier(
            objective="binary",

            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],

            learning_rate=0.1,

            scale_pos_weight=lgbm_scale_pos_weight,

            random_state=42,
            n_jobs=-1,

            verbosity=-1
        )

        model.fit(
            X_fold_train,
            y_fold_train
        )

        val_probability = model.predict_proba(
            X_fold_val
        )[:, 1]

        fold_pr_auc = calculate_pr_auc(
            y_fold_val,
            val_probability
        )

        fold_scores.append(fold_pr_auc)

    mean_pr_auc = float(
        np.mean(fold_scores)
    )

    lgbm_weighted_cv_results.append({
        "params": params,
        "mean_pr_auc": mean_pr_auc,
        "fold_scores": fold_scores
    })


lgbm_weighted_cv_time = (
    time.time() - start_time
)


# ------------------------------------------------------------
# Select best configuration
# ------------------------------------------------------------

lgbm_weighted_best_result = max(
    lgbm_weighted_cv_results,
    key=lambda x: x["mean_pr_auc"]
)

lgbm_weighted_best_params = (
    lgbm_weighted_best_result["params"]
)

lgbm_weighted_best_cv_pr_auc = (
    lgbm_weighted_best_result["mean_pr_auc"]
)


# ------------------------------------------------------------
# Fit best model on FULL TRAINING SET
# ------------------------------------------------------------

lgbm_weighted_best_model = lgb.LGBMClassifier(
    objective="binary",

    n_estimators=(
        lgbm_weighted_best_params["n_estimators"]
    ),

    max_depth=(
        lgbm_weighted_best_params["max_depth"]
    ),

    learning_rate=0.1,

    scale_pos_weight=lgbm_scale_pos_weight,

    random_state=42,
    n_jobs=-1,

    verbosity=-1
)

lgbm_weighted_best_model.fit(
    X_train_lgbm_named,
    y_train_lgbm
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "=== Imbalance-Aware LightGBM "
    "Cross-Validation Complete ===\n"
)

print(
    f"Best n_estimators     : "
    f"{lgbm_weighted_best_params['n_estimators']}"
)

print(
    f"Best max_depth        : "
    f"{lgbm_weighted_best_params['max_depth']}"
)

print(
    f"Best mean CV PR-AUC   : "
    f"{lgbm_weighted_best_cv_pr_auc:.4f}"
)

print(
    f"Cross-validation time : "
    f"{lgbm_weighted_cv_time:.2f} seconds"
)

print("\nAll parameter combinations:")

for result in lgbm_weighted_cv_results:

    params = result["params"]

    print(
        f"n_estimators="
        f"{params['n_estimators']:<3} | "
        f"max_depth="
        f"{params['max_depth']:<2} | "
        f"Mean CV PR-AUC="
        f"{result['mean_pr_auc']:.4f}"
    )

=== Imbalance-Aware LightGBM CV ===
Training non-churn      : 6323
Training churn          : 439
scale_pos_weight        : 14.4032
Parameter combinations  : 4
Cross-validation folds  : 5
Total model fits        : 20
Optimisation metric     : PR-AUC
learning_rate fixed     : 0.1

Starting cross-validation...

=== Imbalance-Aware LightGBM Cross-Validation Complete ===

Best n_estimators     : 50
Best max_depth        : 3
Best mean CV PR-AUC   : 0.0825
Cross-validation time : 5.04 seconds

All parameter combinations:
n_estimators=50  | max_depth=3  | Mean CV PR-AUC=0.0825
n_estimators=100 | max_depth=3  | Mean CV PR-AUC=0.0815
n_estimators=50  | max_depth=6  | Mean CV PR-AUC=0.0768
n_estimators=100 | max_depth=6  | Mean CV PR-AUC=0.0709


### LightGBM Cross-Validation Interpretation

Cross-validation selected relatively shallow LightGBM models for both configurations. The untreated model achieved its highest mean PR-AUC of 0.0812 using 100 estimators with max_depth = 3, while the imbalance-aware model achieved a slightly higher mean PR-AUC of 0.0825 using 50 estimators with max_depth = 3.

The application of scale_pos_weight = 14.4032 therefore produced a modest improvement in cross-validation PR-AUC. However, the difference between the two configurations was small and should not be interpreted as a substantial improvement in predictive performance.

Across both configurations, models with max_depth = 3 generally outperformed those with max_depth = 6. Increasing model complexity therefore did not improve cross-validation performance within the evaluated search space.

The selected untreated and imbalance-aware configurations are subsequently evaluated on the held-out test partition without further hyperparameter adjustment.

### 6.2.4 Tuned LightGBM Evaluation

Following hyperparameter optimisation, the selected untreated and imbalance-aware LightGBM models are evaluated on the held-out test partition. No further changes to model hyperparameters, imbalance treatment or decision threshold are made following this evaluation.

Performance is assessed using the same evaluation framework applied to the preceding classifiers, including accuracy, churn-class precision, recall and F1-score, ROC-AUC, PR-AUC and confusion-matrix values.

In [ ]:
# ============================================================
# Cell 25 — Tuned LightGBM Test Evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


# ------------------------------------------------------------
# Evaluation helper for local Python models
# ------------------------------------------------------------

def evaluate_local_binary_model(
    model,
    X_test,
    y_test
):
    predictions = model.predict(X_test)

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    results = {
        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),

        "Churn Precision": precision_score(
            y_test,
            predictions,
            pos_label=1,
            zero_division=0
        ),

        "Churn Recall": recall_score(
            y_test,
            predictions,
            pos_label=1,
            zero_division=0
        ),

        "Churn F1": f1_score(
            y_test,
            predictions,
            pos_label=1,
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_test,
            probabilities
        ),

        "PR-AUC": calculate_pr_auc(
            y_test,
            probabilities
        ),

        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

    return results, predictions, probabilities


# ------------------------------------------------------------
# Evaluate selected models
# ------------------------------------------------------------

(
    lgbm_tuned_baseline_results,
    lgbm_tuned_baseline_predictions,
    lgbm_tuned_baseline_probabilities
) = evaluate_local_binary_model(
    lgbm_baseline_best_model,
    X_test_lgbm_named,
    y_test_lgbm
)


(
    lgbm_tuned_weighted_results,
    lgbm_tuned_weighted_predictions,
    lgbm_tuned_weighted_probabilities
) = evaluate_local_binary_model(
    lgbm_weighted_best_model,
    X_test_lgbm_named,
    y_test_lgbm
)


# ------------------------------------------------------------
# Display metric comparison
# ------------------------------------------------------------

print("=== Tuned LightGBM Test Evaluation ===\n")

print(
    f"{'Metric':<24}"
    f"{'Baseline':>12}"
    f"{'Imbalance-Aware':>18}"
)

print("-" * 54)

metric_names = [
    "Accuracy",
    "Churn Precision",
    "Churn Recall",
    "Churn F1",
    "ROC-AUC",
    "PR-AUC"
]

for metric in metric_names:

    print(
        f"{metric:<24}"
        f"{lgbm_tuned_baseline_results[metric]:>12.4f}"
        f"{lgbm_tuned_weighted_results[metric]:>18.4f}"
    )


# ------------------------------------------------------------
# Confusion matrices
# ------------------------------------------------------------

print("\n=== Confusion Matrix Counts ===")

print("\nTuned Baseline")

print(
    f"TN: {lgbm_tuned_baseline_results['TN']}   "
    f"FP: {lgbm_tuned_baseline_results['FP']}"
)

print(
    f"FN: {lgbm_tuned_baseline_results['FN']}   "
    f"TP: {lgbm_tuned_baseline_results['TP']}"
)


print("\nTuned Imbalance-Aware")

print(
    f"TN: {lgbm_tuned_weighted_results['TN']}   "
    f"FP: {lgbm_tuned_weighted_results['FP']}"
)

print(
    f"FN: {lgbm_tuned_weighted_results['FN']}   "
    f"TP: {lgbm_tuned_weighted_results['TP']}"
)


# ------------------------------------------------------------
# Selected hyperparameters
# ------------------------------------------------------------

print("\n=== Selected Hyperparameters ===")

print("\nTuned Baseline")

print(
    f"n_estimators   : "
    f"{lgbm_baseline_best_params['n_estimators']}"
)

print(
    f"max_depth      : "
    f"{lgbm_baseline_best_params['max_depth']}"
)

print(
    f"Best CV PR-AUC : "
    f"{lgbm_baseline_best_cv_pr_auc:.4f}"
)

print(
    f"CV time        : "
    f"{lgbm_baseline_cv_time:.2f} seconds"
)


print("\nTuned Imbalance-Aware")

print(
    f"n_estimators      : "
    f"{lgbm_weighted_best_params['n_estimators']}"
)

print(
    f"max_depth         : "
    f"{lgbm_weighted_best_params['max_depth']}"
)

print(
    f"scale_pos_weight  : "
    f"{lgbm_scale_pos_weight:.4f}"
)

print(
    f"Best CV PR-AUC    : "
    f"{lgbm_weighted_best_cv_pr_auc:.4f}"
)

print(
    f"CV time           : "
    f"{lgbm_weighted_cv_time:.2f} seconds"
)


=== Tuned LightGBM Test Evaluation ===

Metric                      Baseline   Imbalance-Aware
------------------------------------------------------
Accuracy                      0.9349            0.5837
Churn Precision               0.0000            0.0875
Churn Recall                  0.0000            0.5727
Churn F1                      0.0000            0.1518
ROC-AUC                       0.5932            0.5762
PR-AUC                        0.0847            0.0834

=== Confusion Matrix Counts ===

Tuned Baseline
TN: 1581   FP: 0
FN: 110   TP: 0

Tuned Imbalance-Aware
TN: 924   FP: 657
FN: 47   TP: 63

=== Selected Hyperparameters ===

Tuned Baseline
n_estimators   : 100
max_depth      : 3
Best CV PR-AUC : 0.0812
CV time        : 2.67 seconds

Tuned Imbalance-Aware
n_estimators      : 50
max_depth         : 3
scale_pos_weight  : 14.4032
Best CV PR-AUC    : 0.0825
CV time           : 5.04 seconds


### Tuned LightGBM Interpretation

The tuned untreated LightGBM model achieved an accuracy of 93.49% but classified every held-out observation as non-churn. Consequently, none of the 110 churn customers were identified, resulting in churn precision, recall and F1-score values of zero. This again demonstrates the limitations of relying on overall accuracy when evaluating the highly imbalanced churn target.

The imbalance-aware LightGBM model identified 63 of the 110 churn customers, corresponding to churn recall of 57.27%. This represented a substantial increase in minority-class detection compared with the untreated model. However, the model also generated 657 false-positive predictions, producing churn precision of 8.75%, an F1-score of 0.1518 and an overall accuracy of 58.37%.

The untreated model achieved a test ROC-AUC of 0.5932 and PR-AUC of 0.0847, compared with 0.5762 and 0.0834 respectively for the imbalance-aware model. Although imbalance treatment produced a small improvement in mean cross-validation PR-AUC, this improvement was not reproduced on the held-out test partition.

Overall, the application of scale_pos_weight substantially increased LightGBM's sensitivity to the churn class but introduced a considerable number of false-positive predictions. The results therefore demonstrate a trade-off between minority-class detection and false-positive control rather than a consistent improvement across all evaluation measures.

# 7. Model Comparison and Selection

This section consolidates the results obtained from Logistic Regression, Decision Tree, Random Forest, XGBoost and LightGBM to enable systematic comparison across the evaluated model families.

Both untreated baseline and imbalance-aware configurations are considered. Model comparison emphasises churn-class precision, recall and F1-score together with ROC-AUC and PR-AUC, while accuracy is retained as contextual information due to the substantial class imbalance.

The purpose of this comparison is not solely to identify the model achieving the highest individual metric. Instead, the results are examined collectively to identify candidate models offering strong predictive performance before model interpretability is incorporated into the final assessment.

## 7.1 Consolidated Model Performance

The held-out test results from all tuned model configurations are consolidated into a single comparison table. This provides a consistent overview of predictive performance and allows the effects of imbalance treatment to be examined across model families.

All reported test results originate from the previously evaluated and subsequently frozen model configurations. No further model tuning or modification is performed during the comparison stage.

In [ ]:
# ============================================================
# Cell 26 — Consolidated Frozen Model Performance
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Recover Logistic Regression best CV PR-AUC values
# from the already-fitted CrossValidator models
# ------------------------------------------------------------

lr_baseline_best_cv_pr_auc = float(
    max(lr_baseline_cv_model.avgMetrics)
)

lr_weighted_best_cv_pr_auc = float(
    max(lr_weighted_cv_model.avgMetrics)
)


# ------------------------------------------------------------
# Helper for constructing comparison rows
# ------------------------------------------------------------

def comparison_row(
    model,
    configuration,
    results,
    cv_pr_auc
):
    return {
        "Model": model,
        "Configuration": configuration,

        "Accuracy": results["Accuracy"],
        "Churn Precision": results["Churn Precision"],
        "Churn Recall": results["Churn Recall"],
        "Churn F1": results["Churn F1"],

        "ROC-AUC": results["ROC-AUC"],
        "PR-AUC": results["PR-AUC"],
        "CV PR-AUC": float(cv_pr_auc),

        "TN": results["TN"],
        "FP": results["FP"],
        "FN": results["FN"],
        "TP": results["TP"]
    }


# ------------------------------------------------------------
# Consolidate all FROZEN model results
# ------------------------------------------------------------

comparison_rows = [

    # Logistic Regression
    comparison_row(
        "Logistic Regression",
        "Baseline",
        lr_tuned_baseline_results,
        lr_baseline_best_cv_pr_auc
    ),

    comparison_row(
        "Logistic Regression",
        "Imbalance-Aware",
        lr_tuned_weighted_results,
        lr_weighted_best_cv_pr_auc
    ),


    # Decision Tree
    comparison_row(
        "Decision Tree",
        "Baseline",
        dt_tuned_baseline_results,
        dt_best_cv_pr_auc
    ),

    comparison_row(
        "Decision Tree",
        "Imbalance-Aware",
        dt_tuned_weighted_results,
        dt_weighted_best_cv_pr_auc
    ),


    # Random Forest
    comparison_row(
        "Random Forest",
        "Baseline",
        rf_tuned_baseline_results,
        rf_best_cv_pr_auc
    ),

    comparison_row(
        "Random Forest",
        "Imbalance-Aware",
        rf_tuned_weighted_results,
        rf_weighted_best_cv_pr_auc
    ),


    # XGBoost
    comparison_row(
        "XGBoost",
        "Baseline",
        xgb_tuned_baseline_results,
        xgb_baseline_best_cv_pr_auc
    ),

    comparison_row(
        "XGBoost",
        "Imbalance-Aware",
        xgb_tuned_weighted_results,
        xgb_weighted_best_cv_pr_auc
    ),


    # LightGBM
    comparison_row(
        "LightGBM",
        "Baseline",
        lgbm_tuned_baseline_results,
        lgbm_baseline_best_cv_pr_auc
    ),

    comparison_row(
        "LightGBM",
        "Imbalance-Aware",
        lgbm_tuned_weighted_results,
        lgbm_weighted_best_cv_pr_auc
    )
]


# ------------------------------------------------------------
# Create comparison DataFrame
# ------------------------------------------------------------

model_comparison_df = pd.DataFrame(
    comparison_rows
)


display_columns = [
    "Model",
    "Configuration",
    "Accuracy",
    "Churn Precision",
    "Churn Recall",
    "Churn F1",
    "ROC-AUC",
    "PR-AUC",
    "CV PR-AUC",
    "TN",
    "FP",
    "FN",
    "TP"
]


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "=== Consolidated Frozen Model Performance ===\n"
)

display(
    model_comparison_df[
        display_columns
    ].round(4)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(model_comparison_df) == 10

assert (
    model_comparison_df["Model"].nunique()
    == 5
)

assert set(
    model_comparison_df["Configuration"]
) == {
    "Baseline",
    "Imbalance-Aware"
}

assert (
    model_comparison_df[
        display_columns[2:]
    ]
    .isnull()
    .sum()
    .sum()
    == 0
)


print("\n=== Comparison Table Validation ===")

print(
    f"Model families       : "
    f"{model_comparison_df['Model'].nunique()}"
)

print(
    f"Model configurations : "
    f"{len(model_comparison_df)}"
)

print("Missing metric values : 0")
print("Frozen results only   : Yes")

print("\nRecovered LR CV PR-AUC:")

print(
    f"Baseline          : "
    f"{lr_baseline_best_cv_pr_auc:.4f}"
)

print(
    f"Imbalance-Aware   : "
    f"{lr_weighted_best_cv_pr_auc:.4f}"
)

=== Consolidated Frozen Model Performance ===



,Model,Configuration,Accuracy,Churn Precision,Churn Recall,Churn F1,ROC-AUC,PR-AUC,CV PR-AUC,TN,FP,FN,TP
0,Logistic Regression,Baseline,0.9349,0.0000,0.0000,0.0000,0.5920,0.0864,0.0848,1581,0,110,0
1,Logistic Regression,Imbalance-Aware,0.6458,0.0863,0.4636,0.1455,0.5881,0.0818,0.0857,1041,540,59,51
2,Decision Tree,Baseline,0.9344,0.0000,0.0000,0.0000,0.4829,0.0653,0.0695,1580,1,110,0
3,Decision Tree,Imbalance-Aware,0.6499,0.0727,0.3727,0.1217,0.5188,0.0769,0.0697,1058,523,69,41
4,Random Forest,Baseline,0.9349,0.0000,0.0000,0.0000,0.5920,0.0879,0.0869,1581,0,110,0
5,Random Forest,Imbalance-Aware,0.5866,0.0787,0.5000,0.1360,0.5912,0.1012,0.0929,937,644,55,55
6,XGBoost,Baseline,0.9349,0.0000,0.0000,0.0000,0.5886,0.0823,0.0826,1581,0,110,0
7,XGBoost,Imbalance-Aware,0.5648,0.0770,0.5182,0.1341,0.5758,0.0854,0.0785,898,683,53,57
8,LightGBM,Baseline,0.9349,0.0000,0.0000,0.0000,0.5932,0.0847,0.0812,1581,0,110,0
9,LightGBM,Imbalance-Aware,0.5837,0.0875,0.5727,0.1518,0.5762,0.0834,0.0825,924,657,47,63



=== Comparison Table Validation ===
Model families       : 5
Model configurations : 10
Missing metric values : 0
Frozen results only   : Yes

Recovered LR CV PR-AUC:
Baseline          : 0.0848
Imbalance-Aware   : 0.0857


## 7.2 Effect of Imbalance-Aware Training

The effect of imbalance-aware training is examined by comparing each model family with its corresponding untreated baseline. Because the baseline classifiers largely favour the majority non-churn class, particular attention is given to changes in churn recall, churn F1-score and PR-AUC.

The comparison quantifies the predictive benefit and cost associated with imbalance treatment, including improvements in minority-class detection alongside any reduction in overall accuracy or increase in false-positive predictions.

In [ ]:
# ============================================================
# Cell 27 — Baseline vs Imbalance-Aware Comparison
# ============================================================

# Split baseline and imbalance-aware rows
baseline_df = (
    model_comparison_df[
        model_comparison_df["Configuration"] == "Baseline"
    ]
    .set_index("Model")
)

imbalance_df = (
    model_comparison_df[
        model_comparison_df["Configuration"] == "Imbalance-Aware"
    ]
    .set_index("Model")
)


# ------------------------------------------------------------
# Calculate metric changes
# ------------------------------------------------------------

imbalance_effect_df = pd.DataFrame({
    "Baseline Accuracy":
        baseline_df["Accuracy"],

    "Imbalance Accuracy":
        imbalance_df["Accuracy"],

    "Δ Accuracy":
        imbalance_df["Accuracy"] - baseline_df["Accuracy"],

    "Baseline Recall":
        baseline_df["Churn Recall"],

    "Imbalance Recall":
        imbalance_df["Churn Recall"],

    "Δ Recall":
        imbalance_df["Churn Recall"] - baseline_df["Churn Recall"],

    "Baseline F1":
        baseline_df["Churn F1"],

    "Imbalance F1":
        imbalance_df["Churn F1"],

    "Δ F1":
        imbalance_df["Churn F1"] - baseline_df["Churn F1"],

    "Baseline PR-AUC":
        baseline_df["PR-AUC"],

    "Imbalance PR-AUC":
        imbalance_df["PR-AUC"],

    "Δ PR-AUC":
        imbalance_df["PR-AUC"] - baseline_df["PR-AUC"],

    "False Positives":
        imbalance_df["FP"],

    "True Positives":
        imbalance_df["TP"]
})


print(
    "=== Effect of Imbalance-Aware Training ===\n"
)

display(
    imbalance_effect_df.round(4)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(imbalance_effect_df) == 5

print("\nComparison complete.")
print("Model families compared : 5")

### Effect of Imbalance-Aware Training Interpretation

Imbalance-aware training substantially changed the classification behaviour of all five model families. Each untreated baseline model produced zero churn recall and zero churn F1-score on the held-out test set, indicating that the high baseline accuracy values were achieved primarily by favouring the majority non-churn class.

Following imbalance treatment, churn recall increased across every model. LightGBM achieved the largest recall improvement, reaching 0.5727 and identifying 63 of the 110 churn customers. XGBoost achieved recall of 0.5182, followed by Random Forest at 0.5000, Logistic Regression at 0.4636 and Decision Tree at 0.3727. LightGBM also achieved the highest churn F1-score of 0.1518.

These improvements were accompanied by substantial reductions in overall accuracy and increases in false-positive predictions. The imbalance-aware models generated between 523 and 683 false positives, demonstrating a clear trade-off between identifying a greater proportion of churn customers and incorrectly classifying non-churn customers as churn.

The effect of imbalance treatment on PR-AUC was not uniform. Random Forest produced the largest improvement, increasing from 0.0879 to 0.1012, while Decision Tree and XGBoost also improved. In contrast, Logistic Regression and LightGBM produced small reductions in held-out PR-AUC despite substantial improvements in churn recall and F1-score.

Overall, imbalance-aware training was necessary to obtain meaningful minority-class classification at the default decision threshold, but it did not produce consistent improvements across all performance measures. The results highlight the importance of evaluating churn models using multiple imbalance-sensitive metrics rather than overall accuracy alone.

## 7.3 Cross-Model Predictive Performance

Following the comparison of untreated and imbalance-aware configurations, cross-model assessment focuses on the imbalance-aware variants because these models demonstrated meaningful detection of the minority churn class.

Performance is compared across churn precision, recall and F1-score together with ROC-AUC and PR-AUC. No single metric is treated as sufficient for determining overall model quality. Instead, the comparison considers the trade-off between identifying churn customers, limiting false-positive predictions and discriminating between the two classes.

In [ ]:
# ============================================================
# Cell 28 — Imbalance-Aware Cross-Model Ranking
# ============================================================

# ------------------------------------------------------------
# Extract imbalance-aware models only
# ------------------------------------------------------------

candidate_models_df = (
    model_comparison_df[
        model_comparison_df["Configuration"]
        == "Imbalance-Aware"
    ]
    .copy()
)


# ------------------------------------------------------------
# Rank models on individual performance measures
# Higher values are better for all metrics below
# ------------------------------------------------------------

candidate_models_df["Precision Rank"] = (
    candidate_models_df["Churn Precision"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

candidate_models_df["Recall Rank"] = (
    candidate_models_df["Churn Recall"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

candidate_models_df["F1 Rank"] = (
    candidate_models_df["Churn F1"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

candidate_models_df["ROC-AUC Rank"] = (
    candidate_models_df["ROC-AUC"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

candidate_models_df["PR-AUC Rank"] = (
    candidate_models_df["PR-AUC"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


# ------------------------------------------------------------
# Display predictive comparison
# ------------------------------------------------------------

ranking_columns = [
    "Model",

    "Churn Precision",
    "Precision Rank",

    "Churn Recall",
    "Recall Rank",

    "Churn F1",
    "F1 Rank",

    "ROC-AUC",
    "ROC-AUC Rank",

    "PR-AUC",
    "PR-AUC Rank",

    "FP",
    "TP"
]


print(
    "=== Imbalance-Aware Model "
    "Performance Ranking ===\n"
)

display(
    candidate_models_df[
        ranking_columns
    ]
    .sort_values(
        "Churn F1",
        ascending=False
    )
    .round(4)
)


# ------------------------------------------------------------
# Metric leaders
# ------------------------------------------------------------

print("\n=== Metric Leaders ===")

for metric in [
    "Churn Precision",
    "Churn Recall",
    "Churn F1",
    "ROC-AUC",
    "PR-AUC"
]:

    best_index = (
        candidate_models_df[metric]
        .idxmax()
    )

    best_model = (
        candidate_models_df.loc[
            best_index,
            "Model"
        ]
    )

    best_value = (
        candidate_models_df.loc[
            best_index,
            metric
        ]
    )

    print(
        f"{metric:<18}: "
        f"{best_model:<20} "
        f"({best_value:.4f})"
    )


print(
    "\nNo composite performance score "
    "has been applied."
)

### Cross-Model Predictive Performance Interpretation

The comparison demonstrates that no single imbalance-aware model dominated across all predictive evaluation measures. LightGBM achieved the highest churn precision (0.0875), recall (0.5727) and F1-score (0.1518), indicating the strongest classification performance at the default decision threshold. It correctly identified 63 of the 110 churn customers, although this was accompanied by 657 false-positive predictions.

Random Forest achieved the highest ROC-AUC (0.5912) and PR-AUC (0.1012), indicating the strongest overall probability-ranking performance among the imbalance-aware models. Its PR-AUC was notably higher than those obtained by the other model families, although its default-threshold churn F1-score of 0.1360 was lower than those of LightGBM and Logistic Regression.

Logistic Regression also remained competitive despite its comparatively simple model structure. It achieved the second-highest churn precision (0.0863) and F1-score (0.1455), together with the second-highest ROC-AUC (0.5881). This result is particularly relevant to the study's research question because Logistic Regression offers greater intrinsic transparency while retaining predictive performance that is competitive with the more complex ensemble models.

XGBoost achieved the second-highest churn recall (0.5182) but did not lead the comparison on the remaining evaluation measures. Decision Tree produced the weakest overall predictive results, ranking last for precision, recall, F1-score, ROC-AUC and PR-AUC.

Overall, the results do not support selecting a model solely on the basis of a single predictive metric. LightGBM, Random Forest and Logistic Regression exhibit different performance advantages and therefore represent particularly relevant candidates when predictive performance is considered alongside model interpretability.

## 7.4 Candidate Model Selection for Explainability Analysis

The predictive comparison identifies LightGBM, Random Forest and Logistic Regression as particularly relevant models for subsequent consideration of the performance–interpretability trade-off.

LightGBM demonstrated the strongest default-threshold churn classification performance, ranking first for churn precision, recall and F1-score. Random Forest demonstrated the strongest probability-ranking performance, achieving the highest ROC-AUC and PR-AUC. Logistic Regression remained competitive in several predictive measures while providing a substantially simpler and more intrinsically transparent model structure.

These results illustrate that predictive superiority depends on the evaluation criterion applied. Consequently, a final model is not selected solely from the predictive results presented in this section. Instead, predictive performance will be considered alongside model explainability and stability in the subsequent analysis.

The remaining model families are retained within the overall comparative evaluation but are not considered leading predictive candidates. XGBoost provided competitive churn recall but did not lead the comparison on other principal metrics, while Decision Tree produced the weakest predictive results across the evaluated measures.

The final assessment will integrate predictive performance with interpretability evidence to determine which model provides the most appropriate overall balance for telecommunications customer churn prediction.

In [ ]:
# ============================================================
# Cell 29 — Save Frozen Model Comparison Artefacts
# ============================================================

import os

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

comparison_path = os.path.join(
    OUTPUTS,
    "model_comparison_frozen_results.csv"
)

imbalance_effect_path = os.path.join(
    OUTPUTS,
    "imbalance_aware_effect_comparison.csv"
)

candidate_ranking_path = os.path.join(
    OUTPUTS,
    "imbalance_aware_model_ranking.csv"
)


# ------------------------------------------------------------
# Save consolidated results
# ------------------------------------------------------------

model_comparison_df.to_csv(
    comparison_path,
    index=False
)

imbalance_effect_df.reset_index().to_csv(
    imbalance_effect_path,
    index=False
)

candidate_models_df[
    ranking_columns
].to_csv(
    candidate_ranking_path,
    index=False
)


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

saved_files = [
    comparison_path,
    imbalance_effect_path,
    candidate_ranking_path
]

print("=== Saved Section 7 Artefacts ===\n")

for path in saved_files:
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0

    print(
        f"{os.path.basename(path):<45} "
        f"Exists: {exists} | "
        f"Size: {size} bytes"
    )


assert all(os.path.exists(path) for path in saved_files)
assert all(os.path.getsize(path) > 0 for path in saved_files)

print("\nAll Section 7 comparison artefacts saved successfully.")